<a href="https://colab.research.google.com/github/tryan01/gravitational-waves/blob/main/microquant_v2_01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import auth
import os

print("Authenticating with Google Cloud...")
auth.authenticate_user()

# Define the bucket and the specific archives to pull
bucket_name = "tryanheider-kalshi-bucket"
archives_to_download = [
    "raw_vault_2026-04-16.tar.gz",
    "raw_vault_2026-04-17.tar.gz",
    "raw_vault_2026-04-18.tar.gz",
    "raw_vault_2026-04-19.tar.gz",
    "raw_vault_2026-04-20.tar.gz",
    "raw_vault_2026-04-22.tar.gz",
    "raw_vault_2026-04-23.tar.gz",
    "raw_vault_2026-04-24.tar.gz",
    "raw_vault_2026-04-25.tar.gz"
]

# Create a local directory in Colab to hold the archives
download_dir = '/content/kalshi_raw_archives'
os.makedirs(download_dir, exist_ok=True)

print(f"Downloading archives from gs://{bucket_name} to {download_dir}...")

for archive in archives_to_download:
    gcs_path = f"gs://{bucket_name}/{archive}"
    local_path = os.path.join(download_dir, archive)

    # Use gsutil for fast, multi-threaded downloading
    print(f"Fetching {archive}...")
    !gcloud storage cp {gcs_path} {local_path}

print("\nDownload complete. Archives available on local Colab disk:")
print(os.listdir(download_dir))

Authenticating with Google Cloud...
Fetching raw_vault_2026-04-16.tar.gz...
Copying gs://tryanheider-kalshi-bucket/raw_vault_2026-04-16.tar.gz to file:///content/kalshi_raw_archives/raw_vault_2026-04-16.tar.gz

Average throughput: 261.1MiB/s
Fetching raw_vault_2026-04-17.tar.gz...
Copying gs://tryanheider-kalshi-bucket/raw_vault_2026-04-17.tar.gz to file:///content/kalshi_raw_archives/raw_vault_2026-04-17.tar.gz

Average throughput: 256.5MiB/s
Fetching raw_vault_2026-04-18.tar.gz...
Copying gs://tryanheider-kalshi-bucket/raw_vault_2026-04-18.tar.gz to file:///content/kalshi_raw_archives/raw_vault_2026-04-18.tar.gz

Average throughput: 216.8MiB/s
Fetching raw_vault_2026-04-19.tar.gz...
Copying gs://tryanheider-kalshi-bucket/raw_vault_2026-04-19.tar.gz to file:///content/kalshi_raw_archives/raw_vault_2026-04-19.tar.gz

Average throughput: 235.5MiB/s
Fetching raw_vault_2026-04-20.tar.gz...
Copying gs://tryanheider-kalshi-bucket/raw_vault_2026-04-20.tar.gz to file:///content/kalshi_raw_arc

In [ ]:
import os
import glob
import tarfile
import ast
import pandas as pd
import numpy as np
import concurrent.futures
from typing import Dict, List, Any, Optional, Tuple

# --- CONFIGURATION ---
DOWNLOAD_DIR = '/content/kalshi_raw_archives'
TARGET_ARCHIVES: List[str] = glob.glob(f"{DOWNLOAD_DIR}/*.tar.gz")
WORKING_DIR: str = '/content/kalshi_working_data'

OUTPUT_DENSE_DIR: str = '/content/kalshi_dense_state_v2'   # For the ML Model
OUTPUT_RAW_DIR: str = '/content/kalshi_clean_raw_v2'       # For the Zero-Assumption Simulator

LEVELS: int = 5
STRIDE_SEC: float = 1.0
WINDOW_HOURS: float = 4.0
WINDOW_SEC: float = WINDOW_HOURS * 3600.0

os.makedirs(WORKING_DIR, exist_ok=True)
os.makedirs(OUTPUT_DENSE_DIR, exist_ok=True)
os.makedirs(OUTPUT_RAW_DIR, exist_ok=True)


def _extract_archive(archive_path: str, extract_path: str) -> None:
    """Extracts a tar archive safely."""
    if not os.path.exists(archive_path):
        print(f"WARNING: Archive not found at {archive_path}")
        return

    archive_name: str = os.path.basename(archive_path).replace('.tar', '').replace('.tar.gz', '')
    target_folder: str = os.path.join(extract_path, archive_name)

    if os.path.exists(target_folder):
        print(f"Archive {archive_name} already extracted. Skipping decompression.")
        return

    print(f"Extracting {archive_path} to {target_folder}...")
    os.makedirs(target_folder, exist_ok=True)
    with tarfile.open(archive_path, 'r:gz') as tar:
        tar.extractall(path=target_folder)
    print(f"Finished extracting {archive_name}.")


def unify_timestamp(ts_series: pd.Series) -> pd.Series:
    """
    Convert mixed timestamp representations into UNIX float seconds.

    Handles:
    - numeric epoch seconds
    - numeric epoch milliseconds
    - ISO strings
    - missing values (preserved as NaN)

    Crucially: this does NOT turn missing timestamps into the huge negative sentinel.
    """
    out = pd.Series(np.nan, index=ts_series.index, dtype=float)

    numeric = pd.to_numeric(ts_series, errors='coerce')
    numeric_mask = numeric.notna()
    if numeric_mask.any():
        numeric_vals = numeric.loc[numeric_mask].astype(float)
        numeric_vals = np.where(numeric_vals > 1e12, numeric_vals / 1000.0, numeric_vals)
        out.loc[numeric_mask] = numeric_vals

    remaining_mask = ~numeric_mask
    if remaining_mask.any():
        dt = pd.to_datetime(ts_series.loc[remaining_mask], format='mixed', errors='coerce', utc=True)
        dt_mask = dt.notna()
        if dt_mask.any():
            out.loc[dt.index[dt_mask]] = dt.loc[dt_mask].astype('int64') / 1e9

    return out


def safe_float(x: Any, default: Optional[float] = np.nan) -> float:
    try:
        if x is None or pd.isna(x):
            return default
        return float(x)
    except Exception:
        return default


def parse_ladder(raw_ladder):
    """
    Parse snapshot ladders stored as nested numpy arrays / lists / tuples / strings.

    Expected shape after normalization:
      [
        ['0.0100', '757201.00'],
        ['0.0200', '4223.00'],
        ...
      ]
    """
    import numpy as np
    import pandas as pd
    import ast

    if raw_ladder is None:
        return []

    try:
        if isinstance(raw_ladder, float) and pd.isna(raw_ladder):
            return []
    except Exception:
        pass

    ladder_obj = raw_ladder

    # pyarrow scalar support
    if hasattr(ladder_obj, "as_py"):
        try:
            ladder_obj = ladder_obj.as_py()
        except Exception:
            pass

    # convert top-level numpy arrays / array-likes
    if hasattr(ladder_obj, "tolist") and not isinstance(ladder_obj, (str, bytes)):
        try:
            ladder_obj = ladder_obj.tolist()
        except Exception:
            pass

    # string representation support
    if isinstance(ladder_obj, str):
        s = ladder_obj.strip()
        if s == "" or s.lower() == "none":
            return []
        try:
            ladder_obj = ast.literal_eval(s)
        except Exception:
            return []

    if not isinstance(ladder_obj, (list, tuple)):
        return []

    out = []
    for item in ladder_obj:
        if hasattr(item, "as_py"):
            try:
                item = item.as_py()
            except Exception:
                pass

        if hasattr(item, "tolist") and not isinstance(item, (str, bytes)):
            try:
                item = item.tolist()
            except Exception:
                pass

        if not isinstance(item, (list, tuple)) or len(item) != 2:
            continue

        try:
            px = float(item[0])
            vol = float(item[1])
        except Exception:
            continue

        if np.isnan(px) or np.isnan(vol) or vol <= 0:
            continue

        out.append((round(px, 2), vol))

    return out


def apply_snapshot_row(row: pd.Series) -> Tuple[Dict[float, float], Dict[float, float]]:
    """
    Build yes-bid / yes-ask books from a snapshot row.

    Observed raw schema:
      yes_dollars_fp -> YES resting bids
      no_dollars_fp  -> NO resting bids, which imply YES asks via ask = 1 - no_bid
    """
    bids: Dict[float, float] = {}
    asks: Dict[float, float] = {}

    yes_ladder = parse_ladder(row.get('yes_dollars_fp'))
    no_ladder = parse_ladder(row.get('no_dollars_fp'))

    for px, vol in yes_ladder:
        if vol > 1e-5:
            bids[round(px, 2)] = float(vol)

    for no_px, vol in no_ladder:
        yes_ask_px = round(1.0 - float(no_px), 2)
        if vol > 1e-5:
            asks[yes_ask_px] = float(vol)

    return bids, asks


def _snapshot_state(
    bids: Dict[float, float],
    asks: Dict[float, float],
    vol_yes: float,
    vol_no: float,
    min_trade: float,
    max_trade: float,
    current_local_ts: float,
    current_exchange_ts: float,
    max_ts: float
) -> Dict[str, float]:
    sorted_bids = sorted(bids.items(), key=lambda x: x[0], reverse=True)
    sorted_asks = sorted(asks.items(), key=lambda x: x[0])

    state = {
        "ts": float(current_exchange_ts) if pd.notna(current_exchange_ts) else np.nan,
        "local_recv_ts": float(current_local_ts),
    }

    for i in range(LEVELS):
        state[f"bid_px_{i+1}"] = float(sorted_bids[i][0]) if i < len(sorted_bids) else 0.0
        state[f"bid_vol_{i+1}"] = float(sorted_bids[i][1]) if i < len(sorted_bids) else 0.0
        state[f"ask_px_{i+1}"] = float(sorted_asks[i][0]) if i < len(sorted_asks) else 1.0
        state[f"ask_vol_{i+1}"] = float(sorted_asks[i][1]) if i < len(sorted_asks) else 0.0

    state["taker_vol_yes"] = vol_yes
    state["taker_vol_no"] = vol_no
    state["window_min_trade_price"] = min_trade
    state["window_max_trade_price"] = max_trade
    state["time_fraction"] = (max_ts - current_local_ts) / WINDOW_SEC

    return state


def process_single_market(market_dir: str) -> str:
    """
    Processes a single market:
    - normalizes timestamps without sentinel corruption
    - saves cleaned raw rows
    - reconstructs the book using true snapshot parsing
    - treats snapshots as hard resets
    - does NOT invalidate on seq jumps inside a single market file
    """
    ticker: str = os.path.basename(market_dir)
    dense_output_path: str = os.path.join(OUTPUT_DENSE_DIR, f"dense_{ticker}.parquet")
    raw_output_path = os.path.join(OUTPUT_RAW_DIR, f"clean_raw_{ticker}.parquet")

    if os.path.exists(dense_output_path) and os.path.exists(raw_output_path):
        return f"[{ticker}] SKIPPED: Both V2 files already exist."

    partition_files: List[str] = glob.glob(f"{market_dir}/*.parquet")
    if not partition_files:
        return f"[{ticker}] SKIPPED: No parquet files."

    try:
        # 1. Concatenate all partitions for this market
        df_list: List[pd.DataFrame] = [pd.read_parquet(f) for f in partition_files]
        master_df: pd.DataFrame = pd.concat(df_list, ignore_index=True)

        # 2. Normalize clocks safely
        if 'ts' in master_df.columns:
            master_df['ts'] = unify_timestamp(master_df['ts'])
        else:
            master_df['ts'] = np.nan

        if 'local_recv_ts' in master_df.columns:
            master_df['local_recv_ts'] = unify_timestamp(master_df['local_recv_ts'])
        else:
            raise ValueError("local_recv_ts missing from raw data")

        # 3. Sort by local receive time; seq is only a stable tie-breaker, not a continuity rule
        if 'seq' not in master_df.columns:
            master_df['seq'] = np.nan
        master_df = master_df.sort_values(by=['local_recv_ts', 'seq'], kind='mergesort').reset_index(drop=True)

        # 4. Save cleaned raw rows for simulator / audits
        master_df.to_parquet(raw_output_path, index=False)

        # 5. Time grid setup
        max_ts: float = float(master_df['local_recv_ts'].max())
        min_ts_clipped: float = max(float(master_df['local_recv_ts'].min()), max_ts - WINDOW_SEC)
        time_grid: np.ndarray = np.arange(min_ts_clipped, max_ts, STRIDE_SEC)

        # 6. Reconstruction state
        resting_bids: Dict[float, float] = {}
        resting_asks: Dict[float, float] = {}

        book_valid: bool = False
        grid_idx: int = 0
        grid_features: List[Dict[str, float]] = []

        rolling_taker_yes: float = 0.0
        rolling_taker_no: float = 0.0
        window_min_trade_price: float = np.inf
        window_max_trade_price: float = -np.inf
        last_exchange_ts: float = np.nan

        # Diagnostics
        snapshot_count: int = 0
        skipped_delta_pre_snapshot_count: int = 0
        empty_snapshot_count: int = 0
        valid_grid_count: int = 0
        total_grid_count: int = 0

        # 7. Walk forward through the full market
        for _, row in master_df.iterrows():
            current_event_ts: float = float(row['local_recv_ts'])

            while grid_idx < len(time_grid) and current_event_ts > time_grid[grid_idx]:
                grid_features.append(_snapshot_state(
                    resting_bids,
                    resting_asks,
                    rolling_taker_yes,
                    rolling_taker_no,
                    window_min_trade_price,
                    window_max_trade_price,
                    float(time_grid[grid_idx]),
                    last_exchange_ts,
                    max_ts
                ))
                total_grid_count += 1
                if book_valid:
                    valid_grid_count += 1

                grid_idx += 1
                rolling_taker_yes, rolling_taker_no = 0.0, 0.0
                window_min_trade_price, window_max_trade_price = np.inf, -np.inf

            if pd.notna(row.get('ts')):
                last_exchange_ts = float(row['ts'])

            event_type = str(row.get('event_type'))

            # -------------------------
            # ORDER BOOK EVENTS
            # -------------------------
            if event_type == 'orderbook_snapshot':
                new_bids, new_asks = apply_snapshot_row(row)
                snapshot_count += 1

                # Ignore terminal / empty snapshots instead of wiping the book
                if len(new_bids) == 0 and len(new_asks) == 0:
                    empty_snapshot_count += 1
                    continue

                resting_bids = new_bids
                resting_asks = new_asks
                book_valid = True
                continue

            elif event_type == 'orderbook_delta':
                if not book_valid:
                    skipped_delta_pre_snapshot_count += 1
                    continue

                tick_side = str(row.get('side'))
                price_val = safe_float(row.get('price_dollars'), default=np.nan)
                delta = safe_float(row.get('delta_fp'), default=0.0)

                if pd.isna(price_val) or tick_side not in ['yes', 'no']:
                    continue

                price_yes = round(price_val, 2) if tick_side == 'yes' else round(1.0 - price_val, 2)
                book = resting_bids if tick_side == 'yes' else resting_asks

                new_vol = book.get(price_yes, 0.0) + delta
                if new_vol > 1e-5:
                    book[price_yes] = new_vol
                else:
                    book.pop(price_yes, None)

            # -------------------------
            # TRADE EVENTS
            # -------------------------
            elif event_type == 'trade':
                taker_side = str(row.get('taker_side'))

                yes_px = safe_float(row.get('yes_price_dollars'), default=np.nan)
                no_px = safe_float(row.get('no_price_dollars'), default=np.nan)

                if not pd.isna(yes_px):
                    price = yes_px
                elif not pd.isna(no_px):
                    price = round(1.0 - no_px, 2)
                else:
                    continue

                trade_count = safe_float(row.get('count_fp'), default=0.0)

                if taker_side == 'yes':
                    rolling_taker_yes += trade_count
                elif taker_side == 'no':
                    rolling_taker_no += trade_count

                if price < window_min_trade_price:
                    window_min_trade_price = price
                if price > window_max_trade_price:
                    window_max_trade_price = price

        # 8. Flush trailing grid points
        while grid_idx < len(time_grid):
            grid_features.append(_snapshot_state(
                resting_bids,
                resting_asks,
                rolling_taker_yes,
                rolling_taker_no,
                window_min_trade_price,
                window_max_trade_price,
                float(time_grid[grid_idx]),
                last_exchange_ts,
                max_ts
            ))
            total_grid_count += 1
            if book_valid:
                valid_grid_count += 1

            grid_idx += 1
            rolling_taker_yes, rolling_taker_no = 0.0, 0.0
            window_min_trade_price, window_max_trade_price = np.inf, -np.inf

        # 9. Save dense output
        df_grid: pd.DataFrame = pd.DataFrame(grid_features)
        if df_grid.empty:
            return f"[{ticker}] FAILED: Output grid empty."

        same_mask = (
            df_grid["ts"].notna()
            & df_grid["local_recv_ts"].notna()
            & (df_grid["ts"].to_numpy() == df_grid["local_recv_ts"].to_numpy())
        )
        if len(df_grid) > 0 and same_mask.all():
            raise ValueError("Clock bug: dense snapshots still have identical ts and local_recv_ts.")

        df_grid.to_parquet(dense_output_path, index=False)

        valid_grid_rate = (valid_grid_count / total_grid_count) if total_grid_count > 0 else 0.0
        return (
            f"[{ticker}] SUCCESS: Wrote Clean Raw & {len(df_grid)} Dense rows | "
            f"snapshots={snapshot_count} | "
            f"empty_snapshots={empty_snapshot_count} | "
            f"skipped_pre_snapshot_deltas={skipped_delta_pre_snapshot_count} | "
            f"valid_grid_rate={valid_grid_rate:.3f}"
        )

    except Exception as e:
        return f"[{ticker}] ERROR: {str(e)}"


if __name__ == '__main__':
    print("--- Starting Pipeline V2 Initialization ---")

    for archive in TARGET_ARCHIVES:
        _extract_archive(archive, WORKING_DIR)

    all_parquets: List[str] = glob.glob(f"{WORKING_DIR}/**/*.parquet", recursive=True)
    if not all_parquets:
        raise FileNotFoundError(f"No .parquet files found in {WORKING_DIR}. Check target archives.")

    market_directories: List[str] = list(set([os.path.dirname(p) for p in all_parquets]))
    print(f"\nDiscovered {len(market_directories)} uncompressed markets. Beginning parallel V2 generation...")

    with concurrent.futures.ProcessPoolExecutor() as executor:
        results: List[str] = list(executor.map(process_single_market, market_directories))

    for r in results:
        print(r)

    print("\nExtraction Complete. Ready for Colab Phase 2.")

--- Starting Pipeline V2 Initialization ---
Extracting /content/kalshi_raw_archives/raw_vault_2026-04-19.tar.gz to /content/kalshi_working_data/raw_vault_2026-04-19.gz...


/tmp/ipykernel_5548/3430026958.py:44: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(path=target_folder)


Finished extracting raw_vault_2026-04-19.gz.
Extracting /content/kalshi_raw_archives/raw_vault_2026-04-25.tar.gz to /content/kalshi_working_data/raw_vault_2026-04-25.gz...
Finished extracting raw_vault_2026-04-25.gz.
Extracting /content/kalshi_raw_archives/raw_vault_2026-04-22.tar.gz to /content/kalshi_working_data/raw_vault_2026-04-22.gz...
Finished extracting raw_vault_2026-04-22.gz.
Extracting /content/kalshi_raw_archives/raw_vault_2026-04-18.tar.gz to /content/kalshi_working_data/raw_vault_2026-04-18.gz...
Finished extracting raw_vault_2026-04-18.gz.
Extracting /content/kalshi_raw_archives/raw_vault_2026-04-24.tar.gz to /content/kalshi_working_data/raw_vault_2026-04-24.gz...
Finished extracting raw_vault_2026-04-24.gz.
Extracting /content/kalshi_raw_archives/raw_vault_2026-04-17.tar.gz to /content/kalshi_working_data/raw_vault_2026-04-17.gz...
Finished extracting raw_vault_2026-04-17.gz.
Extracting /content/kalshi_raw_archives/raw_vault_2026-04-16.tar.gz to /content/kalshi_working_

In [ ]:
# ============================================================
# MicroQuant Stage 1 Dataset: Hot-Density Regime Target
# ============================================================
# Drop-in replacement for KalshiDenseDataset when training a Stage 1
# hot-regime detector.
#
# Core target:
#   For an anchor time t, scan candidate entry offsets e = t, ..., t + candidate_lookahead.
#   At each e, ask whether the then-current bid/ask pair would BOTH fill within
#   fill_lookahead seconds.
#
#   hot_count = number of candidate offsets where both sides fill.
#   label = 1 if hot_count > k_hot else 0.
#
# This is deliberately NOT simulator-PnL. It is a sim-agnostic-ish
# opportunity-density target meant for Stage 1.
#
# Expected model change:
#   model = KalshiTransformer(feature_dim=23, ..., num_classes=2)
#

import os
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset
from typing import List, Dict, Optional, Any


class KalshiHotDensityDataset(Dataset):
    def __init__(
        self,
        file_paths: List[str],
        seq_len: int = 60,
        candidate_lookahead: int = 30,
        fill_lookahead: int = 30,
        stride: int = 2,
        k_hot: int = 5,
        min_spread: float = 0.02,
        inference_mode: bool = False,
        return_diagnostics: bool = False,
        use_grouped_ofi: bool = True,
        explicit_feature_cols: Optional[List[str]] = None,
    ):
        """
        Binary hot-regime dataset.

        Parameters
        ----------
        file_paths:
            Dense parquet files, same as the current KalshiDenseDataset.

        seq_len:
            Number of historical seconds fed to the transformer.

        candidate_lookahead:
            How far forward from anchor t to scan possible future entry times.
            For the first pass, 30 is natural because the best diagnostic policy
            was a 30s hot-window variant.

        fill_lookahead:
            For each candidate entry time e, how far forward to check whether
            the then-current bid and ask both fill. 30 matches the old target.

        stride:
            Dataset sampling stride. Use 2 for training initially; use 1 for
            inference/evaluation if you want maximum temporal coverage.

        k_hot:
            Binary target is hot_count > k_hot. If k_hot=5, then at least 6
            candidate offsets must both-fill. Change the comparison line below
            if you prefer >= semantics.

        min_spread:
            Candidate quote pairs are only counted as good/bad if ask-bid >= min_spread.
            This keeps Stage 1 focused on structurally tradable regimes without
            using full simulator PnL.

        inference_mode:
            If True, __getitem__ returns ticker/timestamps/features, matching your
            existing inference pattern.

        return_diagnostics:
            If True in training mode, __getitem__ returns (features, label, diag_dict).
            Useful for audits, slower for normal training.

        use_grouped_ofi:
            True computes OFI within each ticker file, fixing cross-file shift bleed.
            False mimics the old global-shift behavior more closely.
        """
        self.seq_len = int(seq_len)
        self.candidate_lookahead = int(candidate_lookahead)
        self.fill_lookahead = int(fill_lookahead)
        self.stride = int(stride)
        self.k_hot = int(k_hot)
        self.min_spread = float(min_spread)
        self.inference_mode = bool(inference_mode)
        self.return_diagnostics = bool(return_diagnostics)
        self.use_grouped_ofi = bool(use_grouped_ofi)

        self.bid_col = "bid_px_1"
        self.ask_col = "ask_px_1"
        self.trade_min_col = "window_min_trade_price"
        self.trade_max_col = "window_max_trade_price"

        if explicit_feature_cols is None:
            feature_cols = []
            for level in range(1, 6):
                feature_cols += [
                    f"bid_px_{level}",
                    f"bid_vol_{level}",
                    f"ask_px_{level}",
                    f"ask_vol_{level}",
                ]
            feature_cols += ["taker_vol_yes", "taker_vol_no", "OFI"]
            self.feature_cols = feature_cols
        else:
            self.feature_cols = list(explicit_feature_cols)

        print(f"Loading {len(file_paths)} parquet files...")
        dfs = []
        for f in file_paths:
            df = pd.read_parquet(f)
            df = df.copy()
            df["ticker"] = os.path.basename(f)
            dfs.append(df)

        if not dfs:
            raise ValueError("No dense files supplied.")

        self.data = pd.concat(dfs, ignore_index=True)

        required_cols = {
            "ticker",
            "ts",
            "local_recv_ts",
            self.bid_col,
            self.ask_col,
            self.trade_min_col,
            self.trade_max_col,
        }
        missing_required = required_cols - set(self.data.columns)
        if missing_required:
            raise ValueError(f"Missing required columns: {sorted(missing_required)}")

        # --------------------------------------------------------
        # Feature engineering: OFI + log-volume normalization
        # --------------------------------------------------------
        self._add_ofi()

        missing_features = [c for c in self.feature_cols if c not in self.data.columns]
        if missing_features:
            raise ValueError(f"Missing feature columns after OFI construction: {missing_features}")

        vol_cols = [col for col in self.data.columns if "vol" in col.lower()]
        self.data[vol_cols] = np.log1p(self.data[vol_cols].astype(float))

        # Stable arrays.
        self.ticker_series = self.data["ticker"].astype(str).values
        self.ts_array = self.data["ts"].values
        self.local_recv_ts_array = self.data["local_recv_ts"].values

        self.bid_array = pd.to_numeric(self.data[self.bid_col], errors="coerce").fillna(0.0).values.astype(float)
        self.ask_array = pd.to_numeric(self.data[self.ask_col], errors="coerce").fillna(1.0).values.astype(float)

        self.trade_min_array = (
            pd.to_numeric(self.data[self.trade_min_col], errors="coerce")
            .fillna(np.inf)
            .values
            .astype(float)
        )
        self.trade_max_array = (
            pd.to_numeric(self.data[self.trade_max_col], errors="coerce")
            .fillna(-np.inf)
            .values
            .astype(float)
        )

        self.features_array = self.data[self.feature_cols].astype(float).values
        self.features_array = np.nan_to_num(self.features_array, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)

        # --------------------------------------------------------
        # Label precomputation
        # --------------------------------------------------------
        self._precompute_hot_density_labels()
        self._build_valid_indices()

        print(f"Dataset built with {len(self.indices)} valid samples.")
        self._print_label_summary()

    # ============================================================
    # Feature engineering
    # ============================================================
    def _add_ofi(self) -> None:
        """Adds OFI using either grouped or legacy global shift semantics."""
        if not self.use_grouped_ofi:
            self.data["OFI"] = self._compute_ofi_for_frame(self.data)
            return

        ofi = np.zeros(len(self.data), dtype=float)
        for _, idx in self.data.groupby("ticker", sort=False).indices.items():
            idx = np.asarray(idx)
            ofi[idx] = self._compute_ofi_for_frame(self.data.iloc[idx]).to_numpy(dtype=float)
        self.data["OFI"] = ofi
        self.data["OFI"] = self.data["OFI"].fillna(0.0)

    @staticmethod
    def _compute_ofi_for_frame(df: pd.DataFrame) -> pd.Series:
        bid_px = pd.to_numeric(df["bid_px_1"], errors="coerce")
        bid_vol = pd.to_numeric(df["bid_vol_1"], errors="coerce")
        ask_px = pd.to_numeric(df["ask_px_1"], errors="coerce")
        ask_vol = pd.to_numeric(df["ask_vol_1"], errors="coerce")

        bid_px_prev = bid_px.shift(1)
        bid_vol_prev = bid_vol.shift(1)
        ask_px_prev = ask_px.shift(1)
        ask_vol_prev = ask_vol.shift(1)

        bid_ofi = np.zeros(len(df), dtype=float)
        bid_ofi = np.where(bid_px > bid_px_prev, bid_vol, bid_ofi)
        bid_ofi = np.where(bid_px == bid_px_prev, bid_vol - bid_vol_prev, bid_ofi)
        bid_ofi = np.where(bid_px < bid_px_prev, -bid_vol_prev, bid_ofi)

        ask_ofi = np.zeros(len(df), dtype=float)
        ask_ofi = np.where(ask_px < ask_px_prev, ask_vol, ask_ofi)
        ask_ofi = np.where(ask_px == ask_px_prev, ask_vol - ask_vol_prev, ask_ofi)
        ask_ofi = np.where(ask_px > ask_px_prev, -ask_vol_prev, ask_ofi)

        raw_ofi = bid_ofi - ask_ofi
        out = np.sign(raw_ofi) * np.log1p(np.abs(raw_ofi))
        return pd.Series(out, index=df.index).fillna(0.0)

    # ============================================================
    # Label construction
    # ============================================================
    @staticmethod
    def _forward_min(arr: np.ndarray, horizon: int) -> np.ndarray:
        """At j, min(arr[j+1 : j+horizon+1])."""
        s = pd.Series(arr).shift(-1)
        return s.iloc[::-1].rolling(window=horizon, min_periods=1).min().iloc[::-1].to_numpy(dtype=float)

    @staticmethod
    def _forward_max(arr: np.ndarray, horizon: int) -> np.ndarray:
        """At j, max(arr[j+1 : j+horizon+1])."""
        s = pd.Series(arr).shift(-1)
        return s.iloc[::-1].rolling(window=horizon, min_periods=1).max().iloc[::-1].to_numpy(dtype=float)

    @staticmethod
    def _range_count(binary_arr: np.ndarray, start_offset: int, end_offset: int) -> np.ndarray:
        """
        For each position p, count binary_arr[p+start_offset : p+end_offset+1].
        Offsets are inclusive.
        """
        n = len(binary_arr)
        vals = binary_arr.astype(np.int64)
        csum = np.concatenate([[0], np.cumsum(vals)])
        p = np.arange(n)
        start = np.clip(p + start_offset, 0, n)
        end_excl = np.clip(p + end_offset + 1, 0, n)
        return csum[end_excl] - csum[start]

    def _precompute_hot_density_labels(self) -> None:
        n_total = len(self.data)

        self.candidate_good = np.zeros(n_total, dtype=np.int8)
        self.candidate_bad = np.zeros(n_total, dtype=np.int8)

        self.hot_count_0_30 = np.zeros(n_total, dtype=np.int16)
        self.bad_count_0_30 = np.zeros(n_total, dtype=np.int16)

        self.hot_count_0_10 = np.zeros(n_total, dtype=np.int16)
        self.hot_count_10_20 = np.zeros(n_total, dtype=np.int16)
        self.hot_count_20_30 = np.zeros(n_total, dtype=np.int16)

        self.bad_count_0_10 = np.zeros(n_total, dtype=np.int16)
        self.bad_count_10_20 = np.zeros(n_total, dtype=np.int16)
        self.bad_count_20_30 = np.zeros(n_total, dtype=np.int16)

        self.hot_label = np.zeros(n_total, dtype=np.int64)

        for _, idx in self.data.groupby("ticker", sort=False).indices.items():
            idx = np.asarray(idx)

            bid = self.bid_array[idx]
            ask = self.ask_array[idx]
            trade_min = self.trade_min_array[idx]
            trade_max = self.trade_max_array[idx]

            future_min_trade = self._forward_min(trade_min, self.fill_lookahead)
            future_max_trade = self._forward_max(trade_max, self.fill_lookahead)
            future_min_ask = self._forward_min(ask, self.fill_lookahead)
            future_max_bid = self._forward_max(bid, self.fill_lookahead)

            spread = ask - bid
            tradable = (
                np.isfinite(bid)
                & np.isfinite(ask)
                & (bid > 0.0)
                & (ask < 1.0)
                & (ask > bid)
                & (spread >= self.min_spread)
            )

            bid_fills = future_min_trade < bid
            ask_fills = future_max_trade > ask

            good = tradable & bid_fills & ask_fills

            # Toxic-ish candidate, mirroring the spirit of the old class-0 logic:
            # one side fills, the other does not, and the book reprices against us.
            ask_repriced_down = future_min_ask < ask
            bid_repriced_up = future_max_bid > bid
            bad = tradable & (
                (bid_fills & ~ask_fills & ask_repriced_down)
                | (ask_fills & ~bid_fills & bid_repriced_up)
            )

            good_i = good.astype(np.int8)
            bad_i = bad.astype(np.int8)

            # Full candidate window, default offsets 0..30.
            hot_0_30 = self._range_count(good_i, 0, self.candidate_lookahead)
            bad_0_30 = self._range_count(bad_i, 0, self.candidate_lookahead)

            # Diagnostic timing buckets. If candidate_lookahead < 30 these still work,
            # but later buckets will be clipped by valid-index construction.
            hot_0_10 = self._range_count(good_i, 0, min(10, self.candidate_lookahead))
            hot_10_20 = self._range_count(good_i, 10, min(20, self.candidate_lookahead))
            hot_20_30 = self._range_count(good_i, 20, min(30, self.candidate_lookahead))

            bad_0_10 = self._range_count(bad_i, 0, min(10, self.candidate_lookahead))
            bad_10_20 = self._range_count(bad_i, 10, min(20, self.candidate_lookahead))
            bad_20_30 = self._range_count(bad_i, 20, min(30, self.candidate_lookahead))

            # Primary target: binary hotness from good-count density.
            # User-requested semantics: hot_count > k.
            label = (hot_0_30 > self.k_hot).astype(np.int64)

            self.candidate_good[idx] = good_i
            self.candidate_bad[idx] = bad_i

            self.hot_count_0_30[idx] = hot_0_30.astype(np.int16)
            self.bad_count_0_30[idx] = bad_0_30.astype(np.int16)

            self.hot_count_0_10[idx] = hot_0_10.astype(np.int16)
            self.hot_count_10_20[idx] = hot_10_20.astype(np.int16)
            self.hot_count_20_30[idx] = hot_20_30.astype(np.int16)

            self.bad_count_0_10[idx] = bad_0_10.astype(np.int16)
            self.bad_count_10_20[idx] = bad_10_20.astype(np.int16)
            self.bad_count_20_30[idx] = bad_20_30.astype(np.int16)

            self.hot_label[idx] = label

    def _build_valid_indices(self) -> None:
        """Builds sample anchors whose history and all label lookahead windows stay within ticker."""
        valid_indices = []
        farthest_lookahead = self.candidate_lookahead + self.fill_lookahead
        max_idx = len(self.data) - farthest_lookahead - 1

        for i in range(self.seq_len, max_idx, self.stride):
            history_start = i - self.seq_len
            future_end = i + farthest_lookahead
            if self.ticker_series[history_start] == self.ticker_series[i] == self.ticker_series[future_end]:
                valid_indices.append(i)

        self.indices = np.asarray(valid_indices, dtype=np.int64)

    # ============================================================
    # Dataset API
    # ============================================================
    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        current_idx = int(self.indices[idx])
        history_window = self.features_array[current_idx - self.seq_len : current_idx]
        features = torch.tensor(history_window, dtype=torch.float32)

        if self.inference_mode:
            ticker = str(self.ticker_series[current_idx])
            ts_exchange = float(self.ts_array[current_idx])
            ts_recv = float(self.local_recv_ts_array[current_idx])
            return ticker, ts_exchange, ts_recv, features

        label = torch.tensor(int(self.hot_label[current_idx]), dtype=torch.long)

        if not self.return_diagnostics:
            return features, label

        diag = self.get_diagnostics_for_index(current_idx)
        return features, label, diag

    def get_diagnostics_for_index(self, current_idx: int) -> Dict[str, Any]:
        current_idx = int(current_idx)
        return {
            "ticker": str(self.ticker_series[current_idx]),
            "ts": float(self.ts_array[current_idx]),
            "local_recv_ts": float(self.local_recv_ts_array[current_idx]),
            "label": int(self.hot_label[current_idx]),
            "hot_count_0_30": int(self.hot_count_0_30[current_idx]),
            "bad_count_0_30": int(self.bad_count_0_30[current_idx]),
            "hot_count_0_10": int(self.hot_count_0_10[current_idx]),
            "hot_count_10_20": int(self.hot_count_10_20[current_idx]),
            "hot_count_20_30": int(self.hot_count_20_30[current_idx]),
            "bad_count_0_10": int(self.bad_count_0_10[current_idx]),
            "bad_count_10_20": int(self.bad_count_10_20[current_idx]),
            "bad_count_20_30": int(self.bad_count_20_30[current_idx]),
            "candidate_good_now": int(self.candidate_good[current_idx]),
            "candidate_bad_now": int(self.candidate_bad[current_idx]),
            "bid_px_1": float(self.bid_array[current_idx]),
            "ask_px_1": float(self.ask_array[current_idx]),
            "spread": float(self.ask_array[current_idx] - self.bid_array[current_idx]),
        }

    def diagnostics_frame(self, max_rows: Optional[int] = None) -> pd.DataFrame:
        """Returns diagnostics for valid sample anchors. Useful for class-balance audits."""
        idxs = self.indices if max_rows is None else self.indices[:max_rows]
        rows = [self.get_diagnostics_for_index(int(i)) for i in idxs]
        return pd.DataFrame(rows)

    def _print_label_summary(self) -> None:
        if len(self.indices) == 0:
            print("No valid indices; cannot summarize labels.")
            return

        labels = self.hot_label[self.indices]
        counts = pd.Series(labels).value_counts().sort_index()
        rates = counts / counts.sum()

        print("\nHot-density label distribution:")
        for label_value in [0, 1]:
            c = int(counts.get(label_value, 0))
            r = float(rates.get(label_value, 0.0))
            print(f"  class {label_value}: {c} ({r:.4f})")

        hot_counts = self.hot_count_0_30[self.indices]
        bad_counts = self.bad_count_0_30[self.indices]
        print("\nhot_count_0_30 quantiles:")
        print(pd.Series(hot_counts).quantile([0.50, 0.75, 0.90, 0.95, 0.99]).to_string())
        print("\nbad_count_0_30 quantiles:")
        print(pd.Series(bad_counts).quantile([0.50, 0.75, 0.90, 0.95, 0.99]).to_string())


# ============================================================
# Helper audit functions
# ============================================================

def audit_hot_density_dataset(dataset: KalshiHotDensityDataset, by_ticker: bool = True) -> Dict[str, pd.DataFrame]:
    """Quick diagnostic summaries for the dataset labels and counts."""
    df = dataset.diagnostics_frame()
    out = {}

    label_summary = (
        df.groupby("label")
        .agg(
            samples=("label", "size"),
            mean_hot_count=("hot_count_0_30", "mean"),
            median_hot_count=("hot_count_0_30", "median"),
            mean_bad_count=("bad_count_0_30", "mean"),
            median_bad_count=("bad_count_0_30", "median"),
            mean_spread=("spread", "mean"),
        )
        .reset_index()
    )
    label_summary["rate"] = label_summary["samples"] / len(df)
    out["label_summary"] = label_summary

    if by_ticker:
        ticker_summary = (
            df.groupby("ticker")
            .agg(
                samples=("label", "size"),
                hot_rate=("label", "mean"),
                mean_hot_count=("hot_count_0_30", "mean"),
                mean_bad_count=("bad_count_0_30", "mean"),
            )
            .reset_index()
            .sort_values("hot_rate", ascending=False)
        )
        out["ticker_summary"] = ticker_summary

    print("\n=== Label summary ===")
    print(out["label_summary"].to_string(index=False))

    if by_ticker:
        print("\n=== Ticker summary ===")
        print(out["ticker_summary"].to_string(index=False))

    return out


# ============================================================
# Example usage
# ============================================================
# train_hot_dataset = KalshiHotDensityDataset(
#     file_paths=train_files,
#     seq_len=60,
#     candidate_lookahead=30,
#     fill_lookahead=30,
#     stride=2,
#     k_hot=5,
#     min_spread=0.02,
#     inference_mode=False,
# )
#
# val_hot_dataset = KalshiHotDensityDataset(
#     file_paths=val_files,
#     seq_len=60,
#     candidate_lookahead=30,
#     fill_lookahead=30,
#     stride=2,
#     k_hot=5,
#     min_spread=0.02,
#     inference_mode=False,
# )
#
# audit_hot_density_dataset(train_hot_dataset)
# audit_hot_density_dataset(val_hot_dataset)
#
# train_loader = DataLoader(train_hot_dataset, batch_size=64, shuffle=True, num_workers=2, pin_memory=True)
# val_loader = DataLoader(val_hot_dataset, batch_size=64, shuffle=False, num_workers=2, pin_memory=True)
#
# model = KalshiTransformer(feature_dim=23, d_model=64, nhead=4, num_layers=2, num_classes=2).to(device)


In [ ]:
# edited dataloading for clean data experiment

import os
import glob
from torch.utils.data import DataLoader

print("Scanning local disk for Dense Tensors...")
local_path = '/content/kalshi_dense_state_v2'
all_dense_files = glob.glob(f"{local_path}/*.parquet")

if not all_dense_files:
    raise FileNotFoundError(f"No parquet files found in {local_path}.")

# Group files by Base Game to prevent leakage across dual-market files
market_groups = {}
for file_path in all_dense_files:
    filename = os.path.basename(file_path)
    parts = filename.split('-')
    if len(parts) >= 2:
        base_game = parts[1]   # e.g. '26APR221845NYYBOS'
        market_groups.setdefault(base_game, []).append(file_path)

unique_games = sorted(market_groups.keys())
print(f"Found {len(unique_games)} unique MLB games.")

# Hold out APR 22 games only
APR23_PREFIX = "26APR25"

train_games = [g for g in unique_games if not g.startswith(APR23_PREFIX)]
val_games   = [g for g in unique_games if g.startswith(APR23_PREFIX)]

train_files = [f for game in train_games for f in market_groups[game]]
val_files   = [f for game in val_games for f in market_groups[game]]

print(f"Train Set (non-APR22): {len(train_games)} games -> {len(train_files)} dual-market files")
print(f"Val Set (APR22 only):  {len(val_games)} games -> {len(val_files)} dual-market files")

if len(val_games) == 0:
    raise ValueError("No APR 22 games found in dense files.")
if len(train_games) == 0:
    raise ValueError("No non-APR 22 games found in dense files.")

print("\nBuilding Training Dataset...")
train_dataset = KalshiHotDensityDataset(
        file_paths=train_files,
        seq_len=60,
        candidate_lookahead=30,
        fill_lookahead=30,
        stride=1,
        k_hot=5,
        min_spread=0.02,
        inference_mode=False,
)

print("\nBuilding Validation Dataset...")
val_dataset = KalshiHotDensityDataset(
        file_paths=val_files,
        seq_len=60,
        candidate_lookahead=30,
        fill_lookahead=30,
        stride=1,
        k_hot=5,
        min_spread=0.02,
        inference_mode=False,
)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset, batch_size=64, shuffle=False, num_workers=2, pin_memory=True)

print("\nDataLoaders Ready.")
print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")

Scanning local disk for Dense Tensors...
Found 118 unique MLB games.
Train Set (non-APR22): 103 games -> 206 dual-market files
Val Set (APR22 only):  15 games -> 30 dual-market files

Building Training Dataset...
Loading 206 parquet files...
Dataset built with 2941679 valid samples.

Hot-density label distribution:
  class 0: 2850220 (0.9689)
  class 1: 91459 (0.0311)

hot_count_0_30 quantiles:
0.50     0.0
0.75     0.0
0.90     0.0
0.95     3.0
0.99    12.0

bad_count_0_30 quantiles:
0.50     0.0
0.75     1.0
0.90     8.0
0.95    12.0
0.99    21.0

Building Validation Dataset...
Loading 30 parquet files...
Dataset built with 428399 valid samples.

Hot-density label distribution:
  class 0: 420244 (0.9810)
  class 1: 8155 (0.0190)

hot_count_0_30 quantiles:
0.50    0.0
0.75    0.0
0.90    0.0
0.95    1.0
0.99    9.0

bad_count_0_30 quantiles:
0.50     0.0
0.75     1.0
0.90     6.0
0.95    11.0
0.99    19.0

DataLoaders Ready.
Train batches: 45964 | Val batches: 6694


In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import Tensor

class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 5000):
        super().__init__()
        pe: Tensor = torch.zeros(max_len, d_model)
        position: Tensor = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term: Tensor = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x: Tensor) -> Tensor:
        x = x + self.pe[:, :x.size(1), :]
        return x

class KalshiTransformer(nn.Module):
    def __init__(self, feature_dim: int = 23, d_model: int = 64, nhead: int = 4, num_layers: int = 2, num_classes: int = 3):
        super().__init__()
        self.d_model: int = d_model


        # 1. Input Projection
        self.input_proj = nn.Linear(feature_dim, d_model)
        self.pos_encoder = PositionalEncoding(d_model)

        # 2. Transformer Encoder
        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, batch_first=True, dropout=0.1)
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        # 3. Classification Head
        self.classifier = nn.Sequential(
            nn.Linear(d_model, 32),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(32, num_classes)
        )

    def forward(self, x: Tensor) -> Tensor:

        # Project and encode
        x = self.input_proj(x) * math.sqrt(self.d_model)
        x = self.pos_encoder(x)

        # Pass through attention layers
        x = self.transformer_encoder(x)

        # Extract the final time-step representation
        final_step: Tensor = x[:, -1, :]

        logits: Tensor = self.classifier(final_step)
        return logits

class FocalLoss(nn.Module):
    def __init__(self, alpha: Tensor, gamma: float = 2.0):
        super().__init__()
        self.alpha: Tensor = alpha
        self.gamma: float = gamma

    def forward(self, inputs: Tensor, targets: Tensor) -> Tensor:
        ce_loss: Tensor = F.cross_entropy(inputs, targets, reduction='none')
        pt: Tensor = torch.exp(-ce_loss)
        alpha_t: Tensor = self.alpha.gather(0, targets)
        focal_loss: Tensor = alpha_t * (1 - pt) ** self.gamma * ce_loss
        return focal_loss.mean()

In [ ]:
import os
import json
import numpy as np
import torch
import torch.nn as nn
from sklearn.metrics import confusion_matrix, classification_report, precision_recall_fscore_support
from tqdm import tqdm

# ============================================================
# CONFIG
# ============================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on: {device}")

EPOCHS = 5
LR = 1e-4
WEIGHT_DECAY = 1e-5
GRAD_CLIP = 1.0

SAVE_DIR = "/content/hot_density_checkpoints"
os.makedirs(SAVE_DIR, exist_ok=True)

BEST_VAL_LOSS_PATH = os.path.join(SAVE_DIR, "best_hot_density_val_loss.pth")
BEST_HOT_F1_PATH = os.path.join(SAVE_DIR, "best_hot_density_hot_f1.pth")

# ============================================================
# MODEL: binary hot/cold classifier
# ============================================================

model = KalshiTransformer(
    feature_dim=23,
    d_model=64,
    nhead=4,
    num_layers=2,
    num_classes=2,   # IMPORTANT: binary [0=cold, 1=hot]
).to(device)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY,
)

# ============================================================
# CLASS-WEIGHTED CROSS ENTROPY
# ============================================================

labels = train_dataset.hot_label[train_dataset.indices]
counts = np.bincount(labels, minlength=2)

if np.any(counts == 0):
    raise ValueError(f"One of the classes has zero samples. Counts: {counts}")

weights = counts.sum() / (2.0 * counts)

print("\nClass counts:")
print(f"  class 0 cold: {counts[0]}")
print(f"  class 1 hot:  {counts[1]}")
print("\nClass weights:")
print(f"  class 0 cold: {weights[0]:.4f}")
print(f"  class 1 hot:  {weights[1]:.4f}")

criterion = nn.CrossEntropyLoss(
    weight=torch.tensor(weights, dtype=torch.float32).to(device)
)

# ============================================================
# HELPERS
# ============================================================

def unpack_batch(batch):
    """
    Works whether dataset returns:
        (features, labels)
    or:
        (features, labels, diagnostics)
    """
    if len(batch) == 2:
        batch_features, batch_labels = batch
    elif len(batch) == 3:
        batch_features, batch_labels, _ = batch
    else:
        raise ValueError(f"Unexpected batch length: {len(batch)}")

    return batch_features, batch_labels


def summarize_validation(all_targets, all_preds, all_p_hot):
    all_targets = np.asarray(all_targets)
    all_preds = np.asarray(all_preds)
    all_p_hot = np.asarray(all_p_hot)

    print("\nConfusion Matrix (Rows=true, Cols=pred [0=cold, 1=hot]):")
    cm = confusion_matrix(all_targets, all_preds, labels=[0, 1])
    print(cm)

    print("\nClassification Report:")
    report = classification_report(
        all_targets,
        all_preds,
        labels=[0, 1],
        target_names=["0 cold", "1 hot"],
        zero_division=0,
    )
    print(report)

    precision, recall, f1, support = precision_recall_fscore_support(
        all_targets,
        all_preds,
        labels=[0, 1],
        zero_division=0,
    )

    hot_precision = precision[1]
    hot_recall = recall[1]
    hot_f1 = f1[1]

    print("p_hot quantiles:")
    for q in [0.50, 0.75, 0.90, 0.95, 0.975, 0.99, 0.995, 0.999]:
        print(f"  q={q:>5}: {np.quantile(all_p_hot, q):.6f}")

    print("\nThreshold sweep:")
    for thr in [0.30, 0.40, 0.50, 0.60, 0.70, 0.80, 0.90]:
        mask = all_p_hot >= thr
        n_pred = int(mask.sum())

        if n_pred == 0:
            print(f"  p_hot >= {thr:.2f}: n=0")
            continue

        precision_at_thr = float(all_targets[mask].mean())
        recall_at_thr = float(all_targets[mask].sum() / max(1, all_targets.sum()))

        print(
            f"  p_hot >= {thr:.2f}: "
            f"n={n_pred:6d} | precision={precision_at_thr:.4f} | recall={recall_at_thr:.4f}"
        )

    print("\nTop-N ranking audit:")
    order = np.argsort(-all_p_hot)
    total_hot = max(1, int(all_targets.sum()))

    for n in [100, 300, 500, 700, 1000, 1500, 2000, 5000]:
        n = min(n, len(order))
        idx = order[:n]
        hits = int(all_targets[idx].sum())
        precision_at_n = hits / n
        recall_at_n = hits / total_hot
        min_p = float(all_p_hot[idx[-1]])

        print(
            f"  top {n:5d}: "
            f"hits={hits:5d} | precision={precision_at_n:.4f} | "
            f"recall={recall_at_n:.4f} | cutoff_p_hot={min_p:.6f}"
        )

    return {
        "hot_precision_argmax": float(hot_precision),
        "hot_recall_argmax": float(hot_recall),
        "hot_f1_argmax": float(hot_f1),
        "num_pred_hot_argmax": int((all_preds == 1).sum()),
        "num_true_hot": int((all_targets == 1).sum()),
    }


# ============================================================
# TRAIN LOOP
# ============================================================

best_val_loss = float("inf")
best_hot_f1 = -float("inf")
history = []

for epoch in range(1, EPOCHS + 1):
    print("\n" + "=" * 60)
    print(f"EPOCH {epoch} / {EPOCHS}")
    print("=" * 60)

    # -------------------------
    # TRAIN
    # -------------------------
    model.train()
    train_loss = 0.0
    train_correct = 0
    train_total = 0

    train_bar = tqdm(train_loader, desc="Training", leave=False)

    for batch in train_bar:
        batch_features, batch_labels = unpack_batch(batch)

        batch_features = batch_features.to(device)
        batch_labels = batch_labels.to(device)

        optimizer.zero_grad(set_to_none=True)

        logits = model(batch_features)
        loss = criterion(logits, batch_labels)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=GRAD_CLIP)
        optimizer.step()

        train_loss += loss.item()

        preds = torch.argmax(logits, dim=1)
        train_correct += (preds == batch_labels).sum().item()
        train_total += batch_labels.numel()

        train_bar.set_postfix({
            "loss": f"{loss.item():.4f}",
            "acc": f"{train_correct / max(1, train_total):.4f}",
        })

    avg_train_loss = train_loss / max(1, len(train_loader))
    train_acc = train_correct / max(1, train_total)

    # -------------------------
    # VALIDATE
    # -------------------------
    model.eval()
    val_loss = 0.0

    all_targets = []
    all_preds = []
    all_p_hot = []

    val_bar = tqdm(val_loader, desc="Validating", leave=False)

    with torch.no_grad():
        for batch in val_bar:
            batch_features, batch_labels = unpack_batch(batch)

            batch_features = batch_features.to(device)
            batch_labels = batch_labels.to(device)

            logits = model(batch_features)
            loss = criterion(logits, batch_labels)
            val_loss += loss.item()

            probs = torch.softmax(logits, dim=1)
            p_hot = probs[:, 1]
            preds = torch.argmax(probs, dim=1)

            all_targets.extend(batch_labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())
            all_p_hot.extend(p_hot.cpu().numpy())

    avg_val_loss = val_loss / max(1, len(val_loader))

    print(f"\nTrain Loss: {avg_train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Val Loss:   {avg_val_loss:.4f}")

    metrics = summarize_validation(all_targets, all_preds, all_p_hot)

    epoch_path = os.path.join(SAVE_DIR, f"hot_density_epoch_{epoch}.pth")
    torch.save(model.state_dict(), epoch_path)
    print(f"\nSaved epoch checkpoint: {epoch_path}")

    if avg_val_loss < best_val_loss:
        print(
            f"[+] Val loss improved from {best_val_loss:.4f} "
            f"to {avg_val_loss:.4f}. Saving: {BEST_VAL_LOSS_PATH}"
        )
        best_val_loss = avg_val_loss
        torch.save(model.state_dict(), BEST_VAL_LOSS_PATH)

    if metrics["hot_f1_argmax"] > best_hot_f1:
        print(
            f"[+] Hot F1 improved from {best_hot_f1:.4f} "
            f"to {metrics['hot_f1_argmax']:.4f}. Saving: {BEST_HOT_F1_PATH}"
        )
        best_hot_f1 = metrics["hot_f1_argmax"]
        torch.save(model.state_dict(), BEST_HOT_F1_PATH)

    row = {
        "epoch": epoch,
        "train_loss": float(avg_train_loss),
        "train_acc": float(train_acc),
        "val_loss": float(avg_val_loss),
        **metrics,
    }
    history.append(row)

    with open(os.path.join(SAVE_DIR, "training_history.json"), "w") as f:
        json.dump(history, f, indent=2)

print("\nTraining Complete.")
print(f"Best val-loss checkpoint: {BEST_VAL_LOSS_PATH}")
print(f"Best hot-F1 checkpoint:   {BEST_HOT_F1_PATH}")
print(f"All epoch checkpoints in: {SAVE_DIR}")

Training on: cuda

Class counts:
  class 0 cold: 2850220
  class 1 hot:  91459

Class weights:
  class 0 cold: 0.5160
  class 1 hot:  16.0820

EPOCH 1 / 5



Train Loss: 0.4447 | Train Acc: 0.8409
Val Loss:   0.1893

Confusion Matrix (Rows=true, Cols=pred [0=cold, 1=hot]):
[[396099  24145]
 [  4979   3176]]

Classification Report:
              precision    recall  f1-score   support

      0 cold       0.99      0.94      0.96    420244
       1 hot       0.12      0.39      0.18      8155

    accuracy                           0.93    428399
   macro avg       0.55      0.67      0.57    428399
weighted avg       0.97      0.93      0.95    428399

p_hot quantiles:
  q=  0.5: 0.008593
  q= 0.75: 0.101712
  q=  0.9: 0.348176
  q= 0.95: 0.578788
  q=0.975: 0.757207
  q= 0.99: 0.874517
  q=0.995: 0.912763
  q=0.999: 0.954160

Threshold sweep:
  p_hot >= 0.30: n= 49778 | precision=0.0909 | recall=0.5549
  p_hot >= 0.40: n= 36613 | precision=0.1035 | recall=0.4649
  p_hot >= 0.50: n= 27321 | precision=0.1162 | recall=0.3895
  p_hot >= 0.60: n= 20020 | precision=0.1310 | recall=0.3216
  p_hot >= 0.70: n= 13925 | precision=0.1518 | recall=0.25


Train Loss: 0.4114 | Train Acc: 0.8907
Val Loss:   0.2300

Confusion Matrix (Rows=true, Cols=pred [0=cold, 1=hot]):
[[385348  34896]
 [  4553   3602]]

Classification Report:
              precision    recall  f1-score   support

      0 cold       0.99      0.92      0.95    420244
       1 hot       0.09      0.44      0.15      8155

    accuracy                           0.91    428399
   macro avg       0.54      0.68      0.55    428399
weighted avg       0.97      0.91      0.94    428399

p_hot quantiles:
  q=  0.5: 0.011799
  q= 0.75: 0.161098
  q=  0.9: 0.464432
  q= 0.95: 0.676810
  q=0.975: 0.827116
  q= 0.99: 0.919712
  q=0.995: 0.941802
  q=0.999: 0.966153

Threshold sweep:
  p_hot >= 0.30: n= 67906 | precision=0.0734 | recall=0.6112
  p_hot >= 0.40: n= 51373 | precision=0.0837 | recall=0.5270
  p_hot >= 0.50: n= 38498 | precision=0.0936 | recall=0.4417
  p_hot >= 0.60: n= 28249 | precision=0.1039 | recall=0.3599
  p_hot >= 0.70: n= 19480 | precision=0.1187 | recall=0.28


Train Loss: 0.3777 | Train Acc: 0.9220
Val Loss:   0.1898

Confusion Matrix (Rows=true, Cols=pred [0=cold, 1=hot]):
[[409311  10933]
 [  6574   1581]]

Classification Report:
              precision    recall  f1-score   support

      0 cold       0.98      0.97      0.98    420244
       1 hot       0.13      0.19      0.15      8155

    accuracy                           0.96    428399
   macro avg       0.56      0.58      0.57    428399
weighted avg       0.97      0.96      0.96    428399

p_hot quantiles:
  q=  0.5: 0.002603
  q= 0.75: 0.031841
  q=  0.9: 0.139809
  q= 0.95: 0.308619
  q=0.975: 0.559492
  q= 0.99: 0.879207
  q=0.995: 0.956032
  q=0.999: 0.978299

Threshold sweep:
  p_hot >= 0.30: n= 22022 | precision=0.1082 | recall=0.2922
  p_hot >= 0.40: n= 16416 | precision=0.1182 | recall=0.2380
  p_hot >= 0.50: n= 12514 | precision=0.1263 | recall=0.1939
  p_hot >= 0.60: n=  9701 | precision=0.1381 | recall=0.1643
  p_hot >= 0.70: n=  7551 | precision=0.1459 | recall=0.13


Train Loss: 0.3483 | Train Acc: 0.9418
Val Loss:   0.2194

Confusion Matrix (Rows=true, Cols=pred [0=cold, 1=hot]):
[[402134  18110]
 [  6002   2153]]

Classification Report:
              precision    recall  f1-score   support

      0 cold       0.99      0.96      0.97    420244
       1 hot       0.11      0.26      0.15      8155

    accuracy                           0.94    428399
   macro avg       0.55      0.61      0.56    428399
weighted avg       0.97      0.94      0.96    428399

p_hot quantiles:
  q=  0.5: 0.001348
  q= 0.75: 0.027402
  q=  0.9: 0.200340
  q= 0.95: 0.474722
  q=0.975: 0.767756
  q= 0.99: 0.948322
  q=0.995: 0.974108
  q=0.999: 0.988543

Threshold sweep:
  p_hot >= 0.30: n= 32222 | precision=0.0910 | recall=0.3594
  p_hot >= 0.40: n= 25323 | precision=0.0992 | recall=0.3080
  p_hot >= 0.50: n= 20263 | precision=0.1063 | recall=0.2640
  p_hot >= 0.60: n= 16168 | precision=0.1134 | recall=0.2248
  p_hot >= 0.70: n= 12771 | precision=0.1201 | recall=0.18


Train Loss: 0.3234 | Train Acc: 0.9545
Val Loss:   0.2310

Confusion Matrix (Rows=true, Cols=pred [0=cold, 1=hot]):
[[402565  17679]
 [  6235   1920]]

Classification Report:
              precision    recall  f1-score   support

      0 cold       0.98      0.96      0.97    420244
       1 hot       0.10      0.24      0.14      8155

    accuracy                           0.94    428399
   macro avg       0.54      0.60      0.55    428399
weighted avg       0.97      0.94      0.96    428399

p_hot quantiles:
  q=  0.5: 0.002208
  q= 0.75: 0.036360
  q=  0.9: 0.211780
  q= 0.95: 0.460615
  q=0.975: 0.789490
  q= 0.99: 0.967511
  q=0.995: 0.981157
  q=0.999: 0.991310

Threshold sweep:
  p_hot >= 0.30: n= 32308 | precision=0.0878 | recall=0.3478
  p_hot >= 0.40: n= 24784 | precision=0.0936 | recall=0.2844
  p_hot >= 0.50: n= 19599 | precision=0.0980 | recall=0.2354
  p_hot >= 0.60: n= 15892 | precision=0.1008 | recall=0.1964
  p_hot >= 0.70: n= 12941 | precision=0.1054 | recall=0.16

The simulator code begins here.  An attempt at a near perfect market simulator to efficiently and accurately test and iterate model architecture and execution logic.  Execution logic is also kept below.

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

OUT_DIR = "/content/hot_density_signal_files"
os.makedirs(OUT_DIR, exist_ok=True)

CHECKPOINT = "/content/hot_density_checkpoints/hot_density_epoch_1.pth"
TOP_N_LIST = [300, 500, 700, 1000, 1500, 2000]

SIM_FILES = val_files  # IMPORTANT: set this to the exact APR25 files used for the $1.28 baseline


def load_hot_model(checkpoint_path):
    model = KalshiTransformer(
        feature_dim=23,
        d_model=64,
        nhead=4,
        num_layers=2,
        num_classes=2,
    ).to(device)

    try:
        state = torch.load(checkpoint_path, map_location=device, weights_only=True)
    except TypeError:
        state = torch.load(checkpoint_path, map_location=device)

    model.load_state_dict(state)
    model.eval()
    return model


def generate_hot_outputs(checkpoint_path, file_paths):
    model = load_hot_model(checkpoint_path)

    infer_dataset = KalshiHotDensityDataset(
        file_paths=file_paths,
        seq_len=60,
        candidate_lookahead=30,
        fill_lookahead=30,
        stride=1,
        k_hot=5,
        min_spread=0.02,
        inference_mode=True,
    )

    infer_loader = DataLoader(
        infer_dataset,
        batch_size=256,
        shuffle=False,
        num_workers=2,
        pin_memory=True,
    )

    rows = []

    with torch.no_grad():
        for tickers, ts_exchange_batch, ts_recv_batch, x_batch in tqdm(infer_loader, desc="Hot-density inference"):
            x_batch = x_batch.to(device)
            logits = model(x_batch)
            probs = torch.softmax(logits, dim=1)
            p_hot = probs[:, 1].cpu().numpy()

            ts_exchange_np = (
                ts_exchange_batch.detach().cpu().numpy()
                if torch.is_tensor(ts_exchange_batch)
                else np.asarray(ts_exchange_batch)
            )
            ts_recv_np = (
                ts_recv_batch.detach().cpu().numpy()
                if torch.is_tensor(ts_recv_batch)
                else np.asarray(ts_recv_batch)
            )

            for i, ticker in enumerate(tickers):
                rows.append({
                    "ticker": str(ticker),
                    "ts_exchange": float(ts_exchange_np[i]),
                    "local_recv_ts": float(ts_recv_np[i]),
                    "p_hot": float(p_hot[i]),
                })

    df = pd.DataFrame(rows).sort_values("local_recv_ts").reset_index(drop=True)

    print("\np_hot quantiles:")
    print(df["p_hot"].quantile([0.50, 0.75, 0.90, 0.95, 0.975, 0.99, 0.995, 0.999]))

    return df


def write_top_n_prediction_files(outputs_df, top_n_list):
    paths = []

    for top_n in top_n_list:
        sigs = (
            outputs_df
            .sort_values("p_hot", ascending=False)
            .head(top_n)
            .copy()
            .sort_values("local_recv_ts")
            .reset_index(drop=True)
        )

        sigs["predicted_side"] = "maker"
        sigs["confidence"] = sigs["p_hot"]
        sigs["ts"] = sigs["ts_exchange"]

        # Match your existing simulator prediction schema.
        sigs = sigs[
            [
                "ticker",
                "ts",
                "ts_exchange",
                "local_recv_ts",
                "predicted_side",
                "confidence",
                "p_hot",
            ]
        ]

        path = os.path.join(OUT_DIR, f"predictions_hot_epoch1_top{top_n}.parquet")
        sigs.to_parquet(path, index=False)
        paths.append(path)

        print(f"\nSaved top-{top_n}: {path}")
        print(f"  rows: {len(sigs)}")
        print(f"  unique tickers: {sigs['ticker'].nunique()}")
        print(f"  p_hot cutoff: {sigs['p_hot'].min():.6f}")

    return paths


hot_outputs = generate_hot_outputs(CHECKPOINT, SIM_FILES)
prediction_paths = write_top_n_prediction_files(hot_outputs, TOP_N_LIST)

print("\nPrediction files ready:")
for p in prediction_paths:
    print(p)

Loading 30 parquet files...
Dataset built with 428399 valid samples.

Hot-density label distribution:
  class 0: 420244 (0.9810)
  class 1: 8155 (0.0190)

hot_count_0_30 quantiles:
0.50    0.0
0.75    0.0
0.90    0.0
0.95    1.0
0.99    9.0

bad_count_0_30 quantiles:
0.50     0.0
0.75     1.0
0.90     6.0
0.95    11.0
0.99    19.0


Hot-density inference: 100%|██████████| 1674/1674 [00:21<00:00, 79.10it/s]



p_hot quantiles:
0.500    0.011799
0.750    0.161098
0.900    0.464432
0.950    0.676810
0.975    0.827116
0.990    0.919711
0.995    0.941802
0.999    0.966153
Name: p_hot, dtype: float64

Saved top-300: /content/hot_density_signal_files/predictions_hot_epoch2_top300.parquet
  rows: 300
  unique tickers: 11
  p_hot cutoff: 0.969737

Saved top-500: /content/hot_density_signal_files/predictions_hot_epoch2_top500.parquet
  rows: 500
  unique tickers: 13
  p_hot cutoff: 0.964060

Saved top-700: /content/hot_density_signal_files/predictions_hot_epoch2_top700.parquet
  rows: 700
  unique tickers: 13
  p_hot cutoff: 0.959600

Saved top-1000: /content/hot_density_signal_files/predictions_hot_epoch2_top1000.parquet
  rows: 1000
  unique tickers: 17
  p_hot cutoff: 0.954243

Saved top-1500: /content/hot_density_signal_files/predictions_hot_epoch2_top1500.parquet
  rows: 1500
  unique tickers: 23
  p_hot cutoff: 0.947949

Saved top-2000: /content/hot_density_signal_files/predictions_hot_epoch2_

In [ ]:
import ast
import pandas as pd
import numpy as np
from typing import Dict, Optional, Any, Tuple, List


def safe_float(x: Any, default: Optional[float] = np.nan) -> float:
    try:
        if x is None or pd.isna(x):
            return default
        return float(x)
    except Exception:
        return default


def parse_ladder(raw_ladder: Any) -> List[Tuple[float, float]]:
    """
    Parse snapshot ladders stored as nested numpy arrays / lists / tuples / strings.

    Expected normalized shape:
      [
        ['0.0100', '757201.00'],
        ['0.0200', '4223.00'],
        ...
      ]
    """
    if raw_ladder is None:
        return []

    try:
        if isinstance(raw_ladder, float) and pd.isna(raw_ladder):
            return []
    except Exception:
        pass

    ladder_obj = raw_ladder

    # pyarrow scalar support
    if hasattr(ladder_obj, "as_py"):
        try:
            ladder_obj = ladder_obj.as_py()
        except Exception:
            pass

    # top-level numpy arrays / array-likes
    if hasattr(ladder_obj, "tolist") and not isinstance(ladder_obj, (str, bytes)):
        try:
            ladder_obj = ladder_obj.tolist()
        except Exception:
            pass

    # string representation support
    if isinstance(ladder_obj, str):
        s = ladder_obj.strip()
        if s == "" or s.lower() == "none":
            return []
        try:
            ladder_obj = ast.literal_eval(s)
        except Exception:
            return []

    if not isinstance(ladder_obj, (list, tuple)):
        return []

    out: List[Tuple[float, float]] = []
    for item in ladder_obj:
        if hasattr(item, "as_py"):
            try:
                item = item.as_py()
            except Exception:
                pass

        if hasattr(item, "tolist") and not isinstance(item, (str, bytes)):
            try:
                item = item.tolist()
            except Exception:
                pass

        if not isinstance(item, (list, tuple)) or len(item) != 2:
            continue

        try:
            px = float(item[0])
            vol = float(item[1])
        except Exception:
            continue

        if np.isnan(px) or np.isnan(vol) or vol <= 0:
            continue

        out.append((round(px, 2), vol))

    return out


def apply_snapshot_row(row: dict) -> Tuple[Dict[float, float], Dict[float, float]]:
    """
    Build yes-bid / yes-ask books from a snapshot row.

    Observed schema:
      yes_dollars_fp -> YES resting bids
      no_dollars_fp  -> NO resting bids, which imply YES asks via ask = 1 - no_bid
    """
    bids: Dict[float, float] = {}
    asks: Dict[float, float] = {}

    yes_ladder = parse_ladder(row.get("yes_dollars_fp"))
    no_ladder = parse_ladder(row.get("no_dollars_fp"))

    for px, vol in yes_ladder:
        if vol > 1e-5:
            bids[round(px, 2)] = float(vol)

    for no_px, vol in no_ladder:
        yes_ask_px = round(1.0 - float(no_px), 2)
        if vol > 1e-5:
            asks[yes_ask_px] = float(vol)

    return bids, asks


class KalshiMatchingEngine:
    def __init__(self) -> None:
        self.resting_bids: Dict[float, float] = {}
        self.resting_asks: Dict[float, float] = {}

        # Diagnostics only
        self.snapshot_count: int = 0
        self.empty_snapshot_count: int = 0

        # MAKER UPGRADE
        self.active_orders: Dict[str, Optional[Dict[str, Any]]] = {
            'yes': None,  # Our resting Bid
            'no': None    # Our resting Ask
        }


    def get_best_bid(self) -> Optional[float]:
        valid_bids = [px for px, vol in self.resting_bids.items() if vol > 1e-5]
        best_raw = max(valid_bids) if valid_bids else None
        return best_raw


    def get_best_ask(self) -> Optional[float]:
        valid_asks = [px for px, vol in self.resting_asks.items() if vol > 1e-5]
        best_raw = min(valid_asks) if valid_asks else None
        return best_raw

    """
    def get_best_bid(self) -> Optional[float]:
        valid_bids = [px for px, vol in self.resting_bids.items() if vol > 1e-5]
        best_raw = max(valid_bids) if valid_bids else None

        # Keep your current optimistic assumption for now
        my_order = self.active_orders.get('yes')
        if my_order and my_order['receipt']['status'] == 'pending':
            my_px = my_order['order_price']
            return max(best_raw, my_px) if best_raw is not None else my_px
        return best_raw

    def get_best_ask(self) -> Optional[float]:
        valid_asks = [px for px, vol in self.resting_asks.items() if vol > 1e-5]
        best_raw = min(valid_asks) if valid_asks else None

        # Keep your current optimistic assumption for now
        my_order = self.active_orders.get('no')
        if my_order and my_order['receipt']['status'] == 'pending':
            my_px = my_order['order_price']
            return min(best_raw, my_px) if best_raw is not None else my_px
        return best_raw
        """

    def _refresh_queue_ahead_from_snapshot(self) -> None:
        """
        After a non-empty snapshot reset, refresh queue-ahead estimates
        for any still-pending orders from the authoritative external book.
        """
        yes_order = self.active_orders.get('yes')
        if yes_order is not None and yes_order['receipt']['status'] == 'pending':
            px = yes_order['order_price']
            yes_order['v_ahead'] = self.resting_bids.get(px, 0.0)

        no_order = self.active_orders.get('no')
        if no_order is not None and no_order['receipt']['status'] == 'pending':
            px = no_order['order_price']
            no_order['v_ahead'] = self.resting_asks.get(px, 0.0)

    def process_tick(self, row: dict) -> None:
        """
        The master method called on every tick to maintain the universe.

        Important semantics:
        - orderbook_snapshot: full-ladder reset (unless empty, then ignore)
        - orderbook_delta: single-level liquidity change
        - trade: taker volume crossing the spread
        """
        event_type: str = str(row.get('event_type'))

        if event_type not in ['orderbook_snapshot', 'orderbook_delta', 'trade']:
            return

        # ---------------------------------------------------------
        # 1. Maintain the LOB
        # ---------------------------------------------------------
        if event_type == 'orderbook_snapshot':
            self.snapshot_count += 1

            new_bids, new_asks = apply_snapshot_row(row)

            # Ignore terminal / empty snapshots instead of wiping the book
            if len(new_bids) == 0 and len(new_asks) == 0:
                self.empty_snapshot_count += 1
                return

            self.resting_bids = new_bids
            self.resting_asks = new_asks

            # Refresh queue-ahead estimates from the authoritative snapshot
            self._refresh_queue_ahead_from_snapshot()
            return

        if event_type == 'orderbook_delta':
            tick_side: str = str(row.get('side'))
            price_val = safe_float(row.get('price_dollars'), default=np.nan)
            delta = safe_float(row.get('delta_fp'), default=0.0)

            if pd.isna(price_val) or tick_side not in ['yes', 'no']:
                return

            # YES book directly, NO book becomes YES ask via 1 - price
            price: float = round(price_val, 2) if tick_side == 'yes' else round(1.0 - price_val, 2)
            book = self.resting_bids if tick_side == 'yes' else self.resting_asks

            new_vol: float = book.get(price, 0.0) + delta
            if new_vol > 1e-5:
                book[price] = new_vol
            else:
                book.pop(price, None)

            # Queue Physics: Pulls (DELTA ONLY, not snapshots)
            for o_side, order in self.active_orders.items():
                if order is None or order['receipt']['status'] != 'pending':
                    continue

                if tick_side == o_side and price == order['order_price']:
                    if delta < 0:
                        pulled: float = abs(delta)
                        order['v_ahead'] = max(0.0, order['v_ahead'] - (pulled * 0.5))
                        order['receipt']['pulls_while_waiting'] += pulled

            return

        # ---------------------------------------------------------
        # 2. Trade-driven queue / fill logic
        # ---------------------------------------------------------
        if event_type == 'trade':
            raw_price = row.get('yes_price_dollars')
            if raw_price is None or pd.isna(raw_price):
                return

            price: float = float(raw_price)
            tick_side: str = str(row.get('taker_side'))

            for o_side, order in self.active_orders.items():
                if order is None or order['receipt']['status'] != 'pending':
                    continue

                if float(row['local_recv_ts']) <= order['receipt']['arrival_ts']:
                    continue

                traded: float = float(row.get('count_fp', 0.0))

                # Opposite-side taker flow can hit our resting order
                if tick_side != o_side:
                    is_hit = False
                    if o_side == 'yes' and price <= order['order_price']:
                        is_hit = True
                    elif o_side == 'no' and price >= order['order_price']:
                        is_hit = True

                    if is_hit:
                        order['receipt']['trades_ahead_of_us'] += traded

                        # A. Taker volume must eat the queue ahead of us first
                        if order['v_ahead'] > 0:
                            consumed = min(order['v_ahead'], traded)
                            order['v_ahead'] -= consumed
                            traded -= consumed

                        # B. If we are at the front, the remaining taker volume eats our order
                        if order['v_ahead'] <= 1e-5 and traded > 0:
                            order['my_volume'] -= traded
                            if order['my_volume'] <= 1e-5:
                                order['receipt']['status'] = 'filled'
                                order['receipt']['fill_ts'] = float(row.get('local_recv_ts'))
                                order['receipt']['fill_price'] = order['order_price']

    def submit_limit_order(
        self,
        side: str,
        price: float,
        arrival_ts: float,
        signal_ts: float,
        volume: float = 1.0
    ) -> None:
        """Places an independent order into the YES or NO queue."""
        if side not in ['yes', 'no']:
            return

        order_price = round(price, 2)
        book = self.resting_bids if side == 'yes' else self.resting_asks
        v_ahead = book.get(order_price, 0.0)

        self.active_orders[side] = {
            'order_price': order_price,
            'my_volume': volume,
            'v_ahead': v_ahead,
            'receipt': {
                'signal_ts': signal_ts,
                'arrival_ts': arrival_ts,
                'target_price': order_price,
                'initial_v_ahead': v_ahead,
                'pulls_while_waiting': 0.0,
                'trades_ahead_of_us': 0.0,
                'fill_ts': None,
                'status': 'pending'
            }
        }

    def cancel_active_order(self, side: str, cancel_ts: float) -> Optional[Dict[str, Any]]:
        """Cancels an order on a specific side and returns its receipt."""
        order = self.active_orders.get(side)

        if order is not None and order['receipt']['status'] == 'pending':
            order['receipt']['status'] = 'canceled'
            order['receipt']['exit_ts'] = cancel_ts

            final_receipt = order['receipt'].copy()
            self.active_orders[side] = None
            return final_receipt

        return None

In [ ]:
#newer
class KalshiExecutor:
    def __init__(self, predictions_df: pd.DataFrame):
        # Load and sort predictions chronologically
        self.predictions = predictions_df.sort_values(by='local_recv_ts').to_dict('records')
        self.pred_idx = 0

        # FSM State
        self.state = 'WAITING_FOR_SIGNAL'
        self.active_signal = None
        self.filled_leg = None
        self.entry_receipt = None

        # --- MAKER-MAKER HYPERPARAMETERS ---
        self.LATENCY_PENALTY = 0.020  # 20 milliseconds
        self.WHIPSAW_TIMEOUT = 30.0   # Time to wait for both legs to fill

        # Active Inventory Management (The Chase)
        self.PEG_CYCLE_SEC = 3.0
        self.TTL_PUKE_SEC = 60.0
        self.MIN_SPREAD_BAILOUT = 0.01

        self.whipsaw_start_ts = None
        self.peg_start_ts = None
        self.last_peg_ts = None

        # Final Audit Log
        self.completed_trades = []

    def update(self, row: pd.Series, engine: KalshiMatchingEngine) -> None:
        current_ts = float(row['local_recv_ts'])

        # ---------------------------------------------------------
        # STATE 1: WAITING FOR SIGNAL
        # ---------------------------------------------------------
        if self.state == 'WAITING_FOR_SIGNAL':
            if self.pred_idx >= len(self.predictions):
                return

            next_signal = self.predictions[self.pred_idx]

            if current_ts >= next_signal['local_recv_ts'] + self.LATENCY_PENALTY:
                self.active_signal = next_signal
                self.pred_idx += 1

                best_bid = engine.get_best_bid()
                best_ask = engine.get_best_ask()

                # Require a 2c spread so the Arb is actually profitable after fees
                if best_bid is None or best_ask is None or (best_ask - best_bid) < 0.02:
                    self._log_scratch("Skipped - Vacuum or Spread < 2c")
                    return

                # Deploy the Maker-Maker Net
                engine.submit_limit_order('yes', best_bid, current_ts, self.active_signal['local_recv_ts'])
                engine.submit_limit_order('no', best_ask, current_ts, self.active_signal['local_recv_ts'])

                self.state = 'MAKER_PENDING'
                self.whipsaw_start_ts = current_ts

        # ---------------------------------------------------------
        # STATE 2: MAKER PENDING
        # ---------------------------------------------------------
        elif self.state == 'MAKER_PENDING':
            yes_order = engine.active_orders.get('yes')
            no_order = engine.active_orders.get('no')

            yes_filled = yes_order and yes_order['receipt']['status'] == 'filled'
            no_filled = no_order and no_order['receipt']['status'] == 'filled'

            # 1. Both sides filled
            if yes_filled and no_filled:
                yes_receipt = yes_order['receipt']
                no_receipt = no_order['receipt']

                entry_px = yes_receipt['fill_price']
                exit_px = no_receipt['fill_price']
                pnl = round(exit_px - entry_px, 2)

                self._log_full_whipsaw_trade(
                    yes_receipt=yes_receipt,
                    no_receipt=no_receipt,
                    exit_ts=current_ts,
                    pnl=pnl
                )
                return

            # 2. Timeout: 30 seconds have passed
            if current_ts - self.whipsaw_start_ts > self.WHIPSAW_TIMEOUT:

                # Only cancel if BOTH legs are dead
                if not yes_filled and not no_filled:
                    engine.cancel_active_order('yes', current_ts)
                    engine.cancel_active_order('no', current_ts)
                    self._log_scratch("Scratch - Dead Market, No Fills")
                    return

                # We are legged; move to active pegging
                self.filled_leg = 'yes' if yes_filled else 'no'
                self.entry_receipt = yes_order['receipt'] if yes_filled else no_order['receipt']

                self.state = 'ACTIVE_PEGGING'
                self.peg_start_ts = current_ts
                self.last_peg_ts = current_ts - self.PEG_CYCLE_SEC

        # ---------------------------------------------------------
        # STATE 3: ACTIVE PEGGING
        # ---------------------------------------------------------
        elif self.state == 'ACTIVE_PEGGING':
            exit_side = 'no' if self.filled_leg == 'yes' else 'yes'
            exit_order = engine.active_orders.get(exit_side)

            entry_px = self.entry_receipt['fill_price']
            time_in_peg = current_ts - self.peg_start_ts

            # A. Did our peg get hit naturally?
            if exit_order and exit_order['receipt']['status'] == 'filled':
                exit_px = exit_order['receipt']['fill_price']
                pnl = exit_px - entry_px if self.filled_leg == 'yes' else entry_px - exit_px

                self._log_closed_trade(
                    exit_reason="Scratch/Loss - Peg Hit",
                    exit_px=exit_px,
                    exit_ts=current_ts,
                    pnl=round(pnl, 2),
                    exit_receipt=exit_order['receipt']
                )
                return

            # B. TTL hard puke
            if time_in_peg > self.TTL_PUKE_SEC:
                engine.cancel_active_order(exit_side, current_ts)
                exit_px, pnl = self._execute_market_exit(engine, entry_px, self.filled_leg)
                self._log_closed_trade(
                    exit_reason="Loss - 60s Chase Timeout",
                    exit_px=exit_px,
                    exit_ts=current_ts,
                    pnl=pnl,
                    exit_receipt=None
                )
                return

            # C. The 3-second pegging cycle
            if current_ts - self.last_peg_ts >= self.PEG_CYCLE_SEC:
                self.last_peg_ts = current_ts

                best_bid = engine.get_best_bid()
                best_ask = engine.get_best_ask()

                if best_bid is None or best_ask is None:
                    return

                current_spread = round(best_ask - best_bid, 2)

                # Spread bailout
                if current_spread <= self.MIN_SPREAD_BAILOUT:
                    engine.cancel_active_order(exit_side, current_ts)
                    exit_px, pnl = self._execute_market_exit(engine, entry_px, self.filled_leg)
                    self._log_closed_trade(
                        exit_reason="Loss/Scratch - 1c Spread Bailout",
                        exit_px=exit_px,
                        exit_ts=current_ts,
                        pnl=pnl,
                        exit_receipt=None
                    )
                    return

                # Top-of-book queue retention
                current_order_px = exit_order['order_price'] if exit_order and exit_order['receipt']['status'] == 'pending' else None

                if self.filled_leg == 'yes':
                    if current_order_px is not None and current_order_px <= best_ask:
                        return
                    target_px = round(best_ask - 0.01, 2)
                    if target_px <= best_bid:
                        target_px = round(best_bid + 0.01, 2)
                else:
                    if current_order_px is not None and current_order_px >= best_bid:
                        return
                    target_px = round(best_bid + 0.01, 2)
                    if target_px >= best_ask:
                        target_px = round(best_ask - 0.01, 2)

                engine.cancel_active_order(exit_side, current_ts)
                engine.submit_limit_order(exit_side, target_px, current_ts, current_ts)

    # ---------------------------------------------------------
    # Logging helpers
    # ---------------------------------------------------------
    def _log_scratch(self, reason: str):
        self.completed_trades.append({
            'ticker': self.active_signal['ticker'],
            'signal_side': self.active_signal.get('predicted_side', 'maker'),
            'confidence': self.active_signal.get('confidence'),
            'signal_ts': self.active_signal.get('local_recv_ts'),
            'outcome': reason,
            'entry_price': None,
            'exit_price': None,
            'gross_pnl': 0.0,
            'yes_fill_ts': None,
            'no_fill_ts': None,
            'first_fill_ts': None,
            'second_fill_ts': None,
            'time_between_fills': None,
            'time_from_signal_to_first_fill': None,
            'time_from_order_to_first_fill': None,
            'same_timestamp_fill': None,
        })
        self._reset_fsm()

    def _log_full_whipsaw_trade(self, yes_receipt: dict, no_receipt: dict, exit_ts: float, pnl: float):
        yes_fill_ts = yes_receipt.get('fill_ts')
        no_fill_ts = no_receipt.get('fill_ts')

        first_fill_ts = min(yes_fill_ts, no_fill_ts)
        second_fill_ts = max(yes_fill_ts, no_fill_ts)

        signal_ts = self.active_signal.get('local_recv_ts')
        order_arrival_ts = min(yes_receipt.get('arrival_ts'), no_receipt.get('arrival_ts'))

        time_between_fills = second_fill_ts - first_fill_ts
        time_from_signal_to_first_fill = first_fill_ts - signal_ts if signal_ts is not None else None
        time_from_order_to_first_fill = first_fill_ts - order_arrival_ts if order_arrival_ts is not None else None

        self.completed_trades.append({
            'ticker': self.active_signal['ticker'],
            'signal_side': self.active_signal.get('predicted_side', 'maker'),
            'confidence': self.active_signal.get('confidence'),
            'signal_ts': signal_ts,
            'outcome': "Win - Full Whipsaw Arb",
            'entry_price': yes_receipt['fill_price'],
            'exit_price': no_receipt['fill_price'],
            'gross_pnl': pnl,

            'yes_initial_v_ahead': yes_receipt.get('initial_v_ahead'),
            'no_initial_v_ahead': no_receipt.get('initial_v_ahead'),
            'yes_trades_ahead_of_us': yes_receipt.get('trades_ahead_of_us'),
            'no_trades_ahead_of_us': no_receipt.get('trades_ahead_of_us'),
            'yes_pulls_while_waiting': yes_receipt.get('pulls_while_waiting'),
            'no_pulls_while_waiting': no_receipt.get('pulls_while_waiting'),

            'yes_fill_ts': yes_fill_ts,
            'no_fill_ts': no_fill_ts,
            'first_fill_ts': first_fill_ts,
            'second_fill_ts': second_fill_ts,
            'time_between_fills': time_between_fills,
            'time_from_signal_to_first_fill': time_from_signal_to_first_fill,
            'time_from_order_to_first_fill': time_from_order_to_first_fill,
            'same_timestamp_fill': bool(time_between_fills == 0.0),
            'time_in_trade': time_between_fills,
            'exit_ts': exit_ts,
            'legacy_time_from_yes_fill': exit_ts - yes_fill_ts if yes_fill_ts is not None else None,
        })
        self._reset_fsm()

    def _log_closed_trade(self, exit_reason: str, exit_px: float, exit_ts: float, pnl: float, exit_receipt: dict = None):
        entry_fill_ts = self.entry_receipt.get('fill_ts') if self.entry_receipt is not None else None
        exit_fill_ts = exit_receipt.get('fill_ts') if exit_receipt is not None else exit_ts

        signal_ts = self.active_signal.get('local_recv_ts') if self.active_signal is not None else None
        order_arrival_ts = self.entry_receipt.get('arrival_ts') if self.entry_receipt is not None else None

        time_in_trade = (exit_fill_ts - entry_fill_ts) if (entry_fill_ts is not None and exit_fill_ts is not None) else None
        time_from_signal_to_first_fill = (entry_fill_ts - signal_ts) if (entry_fill_ts is not None and signal_ts is not None) else None
        time_from_order_to_first_fill = (entry_fill_ts - order_arrival_ts) if (entry_fill_ts is not None and order_arrival_ts is not None) else None

        self.completed_trades.append({
            'ticker': self.active_signal['ticker'],
            'signal_side': self.active_signal.get('predicted_side', 'maker'),
            'confidence': self.active_signal.get('confidence'),
            'signal_ts': signal_ts,
            'outcome': exit_reason,
            'entry_price': self.entry_receipt['fill_price'],
            'exit_price': exit_px,
            'gross_pnl': pnl,
            'entry_fill_ts': entry_fill_ts,
            'exit_fill_ts': exit_fill_ts,
            'time_in_trade': time_in_trade,
            'time_from_signal_to_first_fill': time_from_signal_to_first_fill,
            'time_from_order_to_first_fill': time_from_order_to_first_fill,
            'exit_ts': exit_ts,
        })
        self._reset_fsm()

    def _reset_fsm(self):
        self.state = 'WAITING_FOR_SIGNAL'
        self.active_signal = None
        self.entry_receipt = None
        self.filled_leg = None
        self.whipsaw_start_ts = None
        self.peg_start_ts = None
        self.last_peg_ts = None

    def _execute_market_exit(self, engine: KalshiMatchingEngine, entry_px: float, filled_leg: str):
        if filled_leg == 'yes':
            exit_px = engine.get_best_bid()
            if exit_px is None:
                exit_px = 0.01
            pnl = exit_px - entry_px
        else:
            exit_px = engine.get_best_ask()
            if exit_px is None:
                exit_px = 0.99
            pnl = entry_px - exit_px

        return round(exit_px, 2), round(pnl, 2)

In [ ]:
import os
import pandas as pd
from tqdm import tqdm
import time

# --- CONFIGURATION ---
PREDICTIONS_PATH = '/content/hot_density_signal_files/predictions_hot_epoch1_top700.parquet'
RAW_LOCAL_DIR = '/content/kalshi_clean_raw_v2'
OUTPUT_AUDIT_PATH = "/content/hot_density_signal_files/trade_receipts_top700.parquet"

print("=== INITIATING ZERO-ASSUMPTION SIMULATOR ===")

# 1. Load the Signals
df_preds = pd.read_parquet(PREDICTIONS_PATH)
tickers = df_preds['ticker'].unique()
print(f"Loaded {len(df_preds)} signals across {len(tickers)} unique markets.")

all_receipts = []
start_time = time.time()

# 2. The Master Loop
for ticker in tqdm(tickers, desc="Simulating Markets"):
    ticker_preds = df_preds[df_preds['ticker'] == ticker].copy()

    clean_ticker = ticker.replace("dense_", "").replace(".parquet", "")
    raw_file_path = os.path.join(RAW_LOCAL_DIR, f"clean_raw_{clean_ticker}.parquet")

    if not os.path.exists(raw_file_path):
        print(f"\n[!] Raw data missing for {clean_ticker}. Skipping.")
        continue

    df_raw = pd.read_parquet(raw_file_path)

    engine = KalshiMatchingEngine()
    executor = KalshiExecutor(ticker_preds)

    raw_records = df_raw.to_dict('records')

    for row in raw_records:
        engine.process_tick(row)
        executor.update(row, engine)

    all_receipts.extend(executor.completed_trades)

# 3. Generate the Final Audit Log
end_time = time.time()
df_audit = pd.DataFrame(all_receipts)

print("\n=== SIMULATION COMPLETE ===")
print(f"Execution Time: {round(end_time - start_time, 2)} seconds")

if not df_audit.empty:
    df_audit.to_parquet(OUTPUT_AUDIT_PATH, index=False)

    print(f"\nTotal Signals Processed: {len(df_audit)}")
    print("\n--- OUTCOME DISTRIBUTION ---")
    print(df_audit['outcome'].value_counts())

    print("\n--- PNL SUMMARY ---")
    gross_pnl = df_audit['gross_pnl'].sum()
    print(f"Total Gross PnL: ${gross_pnl:.2f}")

    actionable = df_audit[~df_audit['outcome'].str.contains('Skipped')]
    if not actionable.empty:
        print(f"\nActionable trades: {len(actionable)}")
        print(f"Gross PnL / actionable trade: ${actionable['gross_pnl'].mean():.4f}")

    filled_trades = df_audit[~df_audit['outcome'].str.contains('Skipped|Scratch')]
    if not filled_trades.empty:
        print("\n--- SAMPLE COMPLETED TRADES ---")
        print(filled_trades.head())
else:
    print("\n[!] No trades were logged. Check the FSM logic or threshold constraints.")

=== INITIATING ZERO-ASSUMPTION SIMULATOR ===
Loaded 300 signals across 11 unique markets.


Simulating Markets: 100%|██████████| 11/11 [00:35<00:00,  3.20s/it]


=== SIMULATION COMPLETE ===
Execution Time: 35.18 seconds

Total Signals Processed: 300

--- OUTCOME DISTRIBUTION ---
outcome
Skipped - Vacuum or Spread < 2c     215
Scratch - Dead Market, No Fills      47
Win - Full Whipsaw Arb               15
Loss/Scratch - 1c Spread Bailout     11
Scratch/Loss - Peg Hit                8
Loss - 60s Chase Timeout              4
Name: count, dtype: int64

--- PNL SUMMARY ---
Total Gross PnL: $0.15

Actionable trades: 85
Gross PnL / actionable trade: $0.0018

--- SAMPLE COMPLETED TRADES ---
                                           ticker signal_side  confidence  \
4   dense_KXMLBGAME-26APR251415SEASTL-STL.parquet       maker    0.971586   
5   dense_KXMLBGAME-26APR251415SEASTL-STL.parquet       maker    0.978769   
7   dense_KXMLBGAME-26APR251415SEASTL-STL.parquet       maker    0.973269   
9   dense_KXMLBGAME-26APR251415SEASTL-STL.parquet       maker    0.973697   
73  dense_KXMLBGAME-26APR251507CLETOR-CLE.parquet       maker    0.970970   

      

In [ ]:
#volume sim draft starts here

import ast
import pandas as pd
import numpy as np
from typing import Dict, Optional, Any, Tuple, List


EPS = 1e-5
LIVE_ORDER_STATUSES = {"pending", "partial"}


def safe_float(x: Any, default: Optional[float] = np.nan) -> float:
    try:
        if x is None or pd.isna(x):
            return default
        return float(x)
    except Exception:
        return default


def parse_ladder(raw_ladder: Any) -> List[Tuple[float, float]]:
    """
    Parse snapshot ladders stored as nested numpy arrays / lists / tuples / strings.

    Expected normalized shape:
      [
        ['0.0100', '757201.00'],
        ['0.0200', '4223.00'],
        ...
      ]
    """
    if raw_ladder is None:
        return []

    try:
        if isinstance(raw_ladder, float) and pd.isna(raw_ladder):
            return []
    except Exception:
        pass

    ladder_obj = raw_ladder

    if hasattr(ladder_obj, "as_py"):
        try:
            ladder_obj = ladder_obj.as_py()
        except Exception:
            pass

    if hasattr(ladder_obj, "tolist") and not isinstance(ladder_obj, (str, bytes)):
        try:
            ladder_obj = ladder_obj.tolist()
        except Exception:
            pass

    if isinstance(ladder_obj, str):
        s = ladder_obj.strip()
        if s == "" or s.lower() == "none":
            return []
        try:
            ladder_obj = ast.literal_eval(s)
        except Exception:
            return []

    if not isinstance(ladder_obj, (list, tuple)):
        return []

    out: List[Tuple[float, float]] = []
    for item in ladder_obj:
        if hasattr(item, "as_py"):
            try:
                item = item.as_py()
            except Exception:
                pass

        if hasattr(item, "tolist") and not isinstance(item, (str, bytes)):
            try:
                item = item.tolist()
            except Exception:
                pass

        if not isinstance(item, (list, tuple)) or len(item) != 2:
            continue

        try:
            px = float(item[0])
            vol = float(item[1])
        except Exception:
            continue

        if np.isnan(px) or np.isnan(vol) or vol <= 0:
            continue

        out.append((round(px, 2), float(vol)))

    return out


def apply_snapshot_row(row: dict) -> Tuple[Dict[float, float], Dict[float, float]]:
    """
    Build YES-bid / YES-ask books from a snapshot row.

    Observed schema:
      yes_dollars_fp -> YES resting bids
      no_dollars_fp  -> NO resting bids, which imply YES asks via ask = 1 - no_bid
    """
    bids: Dict[float, float] = {}
    asks: Dict[float, float] = {}

    yes_ladder = parse_ladder(row.get("yes_dollars_fp"))
    no_ladder = parse_ladder(row.get("no_dollars_fp"))

    for px, vol in yes_ladder:
        if vol > EPS:
            bids[round(px, 2)] = float(vol)

    for no_px, vol in no_ladder:
        yes_ask_px = round(1.0 - float(no_px), 2)
        if vol > EPS:
            asks[yes_ask_px] = float(vol)

    return bids, asks


def get_yes_trade_price(row: dict) -> Optional[float]:
    """
    Normalize trade price into YES-dollar space.

    Some rows may have yes_price_dollars; others may only have no_price_dollars.
    For no_price_dollars, YES price = 1 - NO price.
    """
    yes_px = safe_float(row.get("yes_price_dollars"), default=np.nan)
    if not pd.isna(yes_px):
        return round(float(yes_px), 2)

    no_px = safe_float(row.get("no_price_dollars"), default=np.nan)
    if not pd.isna(no_px):
        return round(1.0 - float(no_px), 2)

    return None


def get_trade_count(row: dict) -> float:
    count_fp = safe_float(row.get("count_fp"), default=np.nan)
    if not pd.isna(count_fp):
        return float(count_fp)

    count = safe_float(row.get("count"), default=np.nan)
    if not pd.isna(count):
        return float(count)

    return 0.0


def is_live_order(order: Optional[Dict[str, Any]]) -> bool:
    if order is None:
        return False
    return order.get("receipt", {}).get("status") in LIVE_ORDER_STATUSES


class KalshiMatchingEngine:
    def __init__(self, include_own_orders_in_best: bool = False) -> None:
        self.resting_bids: Dict[float, float] = {}
        self.resting_asks: Dict[float, float] = {}

        # Diagnostics only
        self.snapshot_count: int = 0
        self.empty_snapshot_count: int = 0

        # If True, get_best_bid/get_best_ask include our own live resting orders.
        # For exact comparison to older optimistic logic, set True.
        # For more conservative execution simulation, leave False.
        self.include_own_orders_in_best = include_own_orders_in_best

        self.next_order_id: int = 1

        # yes = our YES bid.
        # no  = our YES ask / NO-side resting order.
        self.active_orders: Dict[str, Optional[Dict[str, Any]]] = {
            "yes": None,
            "no": None,
        }

    def get_best_bid(self) -> Optional[float]:
        valid_bids = [px for px, vol in self.resting_bids.items() if vol > EPS]
        best_raw = max(valid_bids) if valid_bids else None

        if self.include_own_orders_in_best:
            my_order = self.active_orders.get("yes")
            if is_live_order(my_order):
                my_px = my_order["order_price"]
                return max(best_raw, my_px) if best_raw is not None else my_px

        return best_raw

    def get_best_ask(self) -> Optional[float]:
        valid_asks = [px for px, vol in self.resting_asks.items() if vol > EPS]
        best_raw = min(valid_asks) if valid_asks else None

        if self.include_own_orders_in_best:
            my_order = self.active_orders.get("no")
            if is_live_order(my_order):
                my_px = my_order["order_price"]
                return min(best_raw, my_px) if best_raw is not None else my_px

        return best_raw

    def get_order_receipt(self, side: str) -> Optional[Dict[str, Any]]:
        order = self.active_orders.get(side)
        if order is None:
            return None
        return order["receipt"].copy()

    def _refresh_queue_ahead_from_snapshot(self) -> None:
        """
        After a non-empty snapshot reset, refresh queue-ahead estimates
        for any still-live orders from the authoritative external book.
        """
        yes_order = self.active_orders.get("yes")
        if is_live_order(yes_order):
            px = yes_order["order_price"]
            yes_order["v_ahead"] = self.resting_bids.get(px, 0.0)

        no_order = self.active_orders.get("no")
        if is_live_order(no_order):
            px = no_order["order_price"]
            no_order["v_ahead"] = self.resting_asks.get(px, 0.0)

    def _apply_fill(self, order: Dict[str, Any], fill_qty: float, fill_ts: float) -> None:
        """
        Apply a partial or full maker fill to one of our resting orders.
        All fills are at the order's resting limit price in this simulator.
        """
        fill_qty = min(float(fill_qty), float(order["remaining_volume"]))
        if fill_qty <= EPS:
            return

        px = float(order["order_price"])

        order["remaining_volume"] -= fill_qty
        order["remaining_volume"] = max(0.0, order["remaining_volume"])
        order["my_volume"] = order["remaining_volume"]  # backward-compatible alias
        order["filled_qty"] += fill_qty

        receipt = order["receipt"]
        receipt["filled_qty"] = order["filled_qty"]
        receipt["remaining_volume"] = order["remaining_volume"]
        receipt["fill_price"] = px
        receipt["avg_fill_price"] = px
        receipt["last_fill_ts"] = fill_ts
        receipt["fill_ts"] = fill_ts  # backward-compatible: completion ts for full fills, latest fill ts for partials

        if receipt["first_fill_ts"] is None:
            receipt["first_fill_ts"] = fill_ts

        receipt["fill_events"].append({
            "ts": fill_ts,
            "qty": fill_qty,
            "price": px,
        })

        if order["remaining_volume"] <= EPS:
            order["remaining_volume"] = 0.0
            order["my_volume"] = 0.0
            receipt["remaining_volume"] = 0.0
            receipt["status"] = "filled"
        else:
            receipt["status"] = "partial"

    def process_tick(self, row: dict) -> None:
        """
        Master method called on every tick.

        Important semantics:
        - orderbook_snapshot: full-ladder reset unless empty, then ignore.
        - orderbook_delta: single-level liquidity change.
        - trade: taker volume crossing the spread and potentially filling our live orders.
        """
        event_type: str = str(row.get("event_type"))

        if event_type not in ["orderbook_snapshot", "orderbook_delta", "trade"]:
            return

        # ---------------------------------------------------------
        # 1. Maintain the LOB
        # ---------------------------------------------------------
        if event_type == "orderbook_snapshot":
            self.snapshot_count += 1

            new_bids, new_asks = apply_snapshot_row(row)

            # Ignore terminal / empty snapshots instead of wiping the book
            if len(new_bids) == 0 and len(new_asks) == 0:
                self.empty_snapshot_count += 1
                return

            self.resting_bids = new_bids
            self.resting_asks = new_asks

            self._refresh_queue_ahead_from_snapshot()
            return

        if event_type == "orderbook_delta":
            tick_side: str = str(row.get("side"))
            price_val = safe_float(row.get("price_dollars"), default=np.nan)
            delta = safe_float(row.get("delta_fp"), default=0.0)

            if pd.isna(price_val) or tick_side not in ["yes", "no"]:
                return

            # YES book directly, NO book becomes YES ask via 1 - no_price
            price: float = round(price_val, 2) if tick_side == "yes" else round(1.0 - price_val, 2)
            book = self.resting_bids if tick_side == "yes" else self.resting_asks

            new_vol: float = book.get(price, 0.0) + delta
            if new_vol > EPS:
                book[price] = new_vol
            else:
                book.pop(price, None)

            # Queue Physics: pulls at our level reduce estimated volume ahead.
            # Heuristic retained from prior engine.
            for o_side, order in self.active_orders.items():
                if not is_live_order(order):
                    continue

                if tick_side == o_side and price == order["order_price"] and delta < 0:
                    pulled: float = abs(delta)
                    order["v_ahead"] = max(0.0, order["v_ahead"] - (pulled * 0.5))
                    order["receipt"]["pulls_while_waiting"] += pulled

            return

        # ---------------------------------------------------------
        # 2. Trade-driven queue / fill logic
        # ---------------------------------------------------------
        if event_type == "trade":
            price = get_yes_trade_price(row)
            if price is None:
                return

            tick_side: str = str(row.get("taker_side"))
            if tick_side not in ["yes", "no"]:
                return

            current_ts = safe_float(row.get("local_recv_ts"), default=np.nan)
            if pd.isna(current_ts):
                return

            traded: float = get_trade_count(row)
            if traded <= EPS:
                return

            for o_side, order in self.active_orders.items():
                if not is_live_order(order):
                    continue

                # Cannot fill before our order has arrived.
                if current_ts <= order["receipt"]["arrival_ts"]:
                    continue

                # Opposite-side taker flow can hit our resting order.
                if tick_side == o_side:
                    continue

                is_hit = False

                # Our YES bid gets hit by NO-side taker flow selling YES down into bid.
                if o_side == "yes" and price <= order["order_price"]:
                    is_hit = True

                # Our YES ask / NO-side order gets hit by YES-side taker flow buying YES up into ask.
                elif o_side == "no" and price >= order["order_price"]:
                    is_hit = True

                if not is_hit:
                    continue

                remaining_taker_volume = traded
                order["receipt"]["raw_hit_volume_at_or_through_price"] += traded

                # A. Taker volume eats queue ahead first.
                if order["v_ahead"] > EPS:
                    consumed = min(order["v_ahead"], remaining_taker_volume)
                    order["v_ahead"] -= consumed
                    remaining_taker_volume -= consumed
                    order["receipt"]["trades_ahead_of_us"] += consumed

                # B. Remaining taker volume fills us, possibly partially.
                if order["v_ahead"] <= EPS and remaining_taker_volume > EPS:
                    self._apply_fill(order, remaining_taker_volume, float(current_ts))

    def submit_limit_order(
        self,
        side: str,
        price: float,
        arrival_ts: float,
        signal_ts: float,
        volume: float = 1.0,
        tag: str = "entry",
    ) -> Optional[int]:
        """
        Places an independent order into the YES or NO queue.

        side='yes': our YES bid.
        side='no' : our YES ask / NO-side resting order.
        """
        if side not in ["yes", "no"]:
            return None

        order_price = round(float(price), 2)
        volume = float(volume)

        if volume <= EPS:
            return None

        book = self.resting_bids if side == "yes" else self.resting_asks
        v_ahead = float(book.get(order_price, 0.0))

        order_id = self.next_order_id
        self.next_order_id += 1

        self.active_orders[side] = {
            "order_id": order_id,
            "side": side,
            "tag": tag,
            "order_price": order_price,
            "initial_volume": volume,
            "remaining_volume": volume,
            "my_volume": volume,  # backward-compatible alias
            "filled_qty": 0.0,
            "v_ahead": v_ahead,
            "receipt": {
                "order_id": order_id,
                "side": side,
                "tag": tag,
                "signal_ts": signal_ts,
                "arrival_ts": arrival_ts,
                "target_price": order_price,
                "initial_volume": volume,
                "initial_v_ahead": v_ahead,
                "pulls_while_waiting": 0.0,
                "trades_ahead_of_us": 0.0,
                "raw_hit_volume_at_or_through_price": 0.0,
                "filled_qty": 0.0,
                "remaining_volume": volume,
                "fill_price": None,
                "avg_fill_price": None,
                "first_fill_ts": None,
                "last_fill_ts": None,
                "fill_ts": None,
                "fill_events": [],
                "status": "pending",
            },
        }

        return order_id

    def cancel_active_order(self, side: str, cancel_ts: float) -> Optional[Dict[str, Any]]:
        """
        Cancels a live order and returns its receipt.

        If the order was partially filled, the returned receipt preserves the fill history.
        """
        order = self.active_orders.get(side)

        if not is_live_order(order):
            return None

        receipt = order["receipt"]
        receipt["exit_ts"] = cancel_ts
        receipt["cancel_ts"] = cancel_ts
        receipt["filled_qty"] = order["filled_qty"]
        receipt["remaining_volume"] = order["remaining_volume"]

        if order["filled_qty"] > EPS:
            receipt["status"] = "canceled_partial"
        else:
            receipt["status"] = "canceled_unfilled"

        final_receipt = receipt.copy()
        self.active_orders[side] = None
        return final_receipt

In [ ]:
class KalshiExecutor:
    def __init__(self, predictions_df: pd.DataFrame, order_size: float = 1.0):
        # Load and sort predictions chronologically
        self.predictions = predictions_df.sort_values(by="local_recv_ts").to_dict("records")
        self.pred_idx = 0

        self.ORDER_SIZE = float(order_size)

        # FSM State
        self.state = "WAITING_FOR_SIGNAL"
        self.active_signal = None
        self.MAX_SIGNAL_AGE_SEC = 30.0 #temporary, simul track multiple trades later

        # --- MAKER-MAKER HYPERPARAMETERS ---
        self.LATENCY_PENALTY = 0.020
        self.WHIPSAW_TIMEOUT = 30.0

        # Active Inventory Management
        self.PEG_CYCLE_SEC = 3.0
        self.TTL_PUKE_SEC = 60.0
        self.MIN_SPREAD_BAILOUT = 0.01

        self.whipsaw_start_ts = None
        self.peg_start_ts = None
        self.last_peg_ts = None

        # Entry summary after timeout
        self.entry_yes_receipt = None
        self.entry_no_receipt = None
        self.entry_yes_px = None
        self.entry_no_px = None

        self.yes_filled_qty = 0.0
        self.no_filled_qty = 0.0
        self.matched_qty = 0.0
        self.net_yes_qty = 0.0

        self.clean_pnl = 0.0

        # Rescue / pegging state
        self.rescue_side = None
        self.rescue_qty_target = 0.0
        self.rescue_qty_filled = 0.0
        self.rescue_pnl = 0.0
        self.rescue_exit_notional = 0.0
        self.rescue_last_fill_ts = None
        self.rescue_order_counted_qty = {}

        # Final Audit Log
        self.completed_trades = []

    # ============================================================
    # Main update loop
    # ============================================================
    def update(self, row: pd.Series, engine: KalshiMatchingEngine) -> None:
        current_ts = float(row["local_recv_ts"])

        # ---------------------------------------------------------
        # STATE 1: WAITING FOR SIGNAL
        # ---------------------------------------------------------
        if self.state == "WAITING_FOR_SIGNAL":
            if self.pred_idx >= len(self.predictions):
                return

            # Consume and explicitly skip stale signals until we find either:
            #   A. a future signal not ready yet, or
            #   B. a fresh actionable signal.
            while self.pred_idx < len(self.predictions):
                next_signal = self.predictions[self.pred_idx]
                signal_ts = float(next_signal["local_recv_ts"])
                actionable_ts = signal_ts + self.LATENCY_PENALTY

                # Not ready yet.
                if current_ts < actionable_ts:
                    return

                signal_age = current_ts - actionable_ts

                # Too old to act on.
                if signal_age > self.MAX_SIGNAL_AGE_SEC:
                    self.active_signal = next_signal
                    self.pred_idx += 1
                    self._log_scratch("Skipped - Stale Signal")
                    continue

                # Fresh signal.
                break

            if self.pred_idx >= len(self.predictions):
                return

            next_signal = self.predictions[self.pred_idx]
            self.active_signal = next_signal
            self.pred_idx += 1

            best_bid = engine.get_best_bid()
            best_ask = engine.get_best_ask()

            if best_bid is None or best_ask is None:
                self._log_scratch("Skipped - Vacuum")
                return

            spread = round(best_ask - best_bid, 2)

            if spread < 0.02:
                self._log_scratch("Skipped - Spread < 2c")
                return

            engine.submit_limit_order(
                side="yes",
                price=best_bid,
                arrival_ts=current_ts,
                signal_ts=self.active_signal["local_recv_ts"],
                volume=self.ORDER_SIZE,
                tag="entry_yes",
            )

            engine.submit_limit_order(
                side="no",
                price=best_ask,
                arrival_ts=current_ts,
                signal_ts=self.active_signal["local_recv_ts"],
                volume=self.ORDER_SIZE,
                tag="entry_no",
            )

            self.state = "MAKER_PENDING"
            self.whipsaw_start_ts = current_ts

        # ---------------------------------------------------------
        # STATE 2: MAKER PENDING
        # ---------------------------------------------------------
        elif self.state == "MAKER_PENDING":
            yes_order = engine.active_orders.get("yes")
            no_order = engine.active_orders.get("no")

            yes_qty = self._order_filled_qty(yes_order)
            no_qty = self._order_filled_qty(no_order)

            # Case A: full clean whipsaw at requested size.
            if yes_qty >= self.ORDER_SIZE - EPS and no_qty >= self.ORDER_SIZE - EPS:
                yes_receipt = engine.get_order_receipt("yes")
                no_receipt = engine.get_order_receipt("no")

                self._load_entry_summary(yes_receipt, no_receipt)
                self._log_terminal_trade(
                    outcome="Win - Full Clean Whipsaw",
                    exit_reason="both_entry_orders_filled",
                    exit_ts=current_ts,
                )
                return

            # Case B: whipsaw window expired.
            if current_ts - self.whipsaw_start_ts > self.WHIPSAW_TIMEOUT:
                # Snapshot receipts before canceling.
                yes_receipt_before = engine.get_order_receipt("yes")
                no_receipt_before = engine.get_order_receipt("no")

                # Cancel live remainders.
                yes_cancel_receipt = engine.cancel_active_order("yes", current_ts)
                no_cancel_receipt = engine.cancel_active_order("no", current_ts)

                yes_receipt = yes_cancel_receipt or yes_receipt_before
                no_receipt = no_cancel_receipt or no_receipt_before

                self._load_entry_summary(yes_receipt, no_receipt)

                # No fills at all.
                if self.yes_filled_qty <= EPS and self.no_filled_qty <= EPS:
                    self._log_scratch("Scratch - Dead Market, No Fills")
                    return

                # Some matched quantity, but no net inventory.
                if abs(self.net_yes_qty) <= EPS:
                    if self.matched_qty >= self.ORDER_SIZE - EPS:
                        outcome = "Win - Full Clean Whipsaw"
                    else:
                        outcome = "Win - Partial Clean Whipsaw"

                    self._log_terminal_trade(
                        outcome=outcome,
                        exit_reason="timeout_no_net_inventory",
                        exit_ts=current_ts,
                    )
                    return

                # We have inventory to rescue.
                self.state = "ACTIVE_PEGGING"
                self.peg_start_ts = current_ts
                self.last_peg_ts = current_ts - self.PEG_CYCLE_SEC

                self.rescue_side = "no" if self.net_yes_qty > 0 else "yes"
                self.rescue_qty_target = abs(self.net_yes_qty)
                self.rescue_qty_filled = 0.0
                self.rescue_pnl = 0.0
                self.rescue_exit_notional = 0.0
                self.rescue_last_fill_ts = None
                self.rescue_order_counted_qty = {}

                placed = self._place_or_reprice_rescue_order(
                    engine=engine,
                    current_ts=current_ts,
                    force=True,
                )

                if not placed:
                    self._market_exit_remaining_inventory(
                        engine=engine,
                        current_ts=current_ts,
                        reason="Loss/Scratch - Could Not Place Initial Rescue Peg",
                    )
                    return

        # ---------------------------------------------------------
        # STATE 3: ACTIVE PEGGING
        # ---------------------------------------------------------
        elif self.state == "ACTIVE_PEGGING":
            self._sync_rescue_progress(engine)

            # If rescue order fully handled the imbalance, close.
            if self.rescue_qty_filled >= self.rescue_qty_target - EPS:
                self._log_terminal_trade(
                    outcome=self._classify_rescue_outcome("Closed - Rescue Peg Filled"),
                    exit_reason="rescue_peg_filled",
                    exit_ts=current_ts,
                )
                return

            time_in_peg = current_ts - self.peg_start_ts

            # Hard TTL puke.
            if time_in_peg > self.TTL_PUKE_SEC:
                self._market_exit_remaining_inventory(
                    engine=engine,
                    current_ts=current_ts,
                    reason="Loss - Chase Timeout Market Exit",
                )
                return

            # Pegging cycle.
            if current_ts - self.last_peg_ts >= self.PEG_CYCLE_SEC:
                self.last_peg_ts = current_ts

                best_bid = engine.get_best_bid()
                best_ask = engine.get_best_ask()

                if best_bid is None or best_ask is None:
                    return

                current_spread = round(best_ask - best_bid, 2)

                # Spread bailout.
                if current_spread <= self.MIN_SPREAD_BAILOUT:
                    self._market_exit_remaining_inventory(
                        engine=engine,
                        current_ts=current_ts,
                        reason="Loss/Scratch - 1c Spread Bailout",
                    )
                    return

                self._place_or_reprice_rescue_order(
                    engine=engine,
                    current_ts=current_ts,
                    force=False,
                )

    # ============================================================
    # Quantity helpers
    # ============================================================
    def _order_filled_qty(self, order) -> float:
        if order is None:
            return 0.0
        return float(order.get("filled_qty", order.get("receipt", {}).get("filled_qty", 0.0)) or 0.0)

    def _receipt_filled_qty(self, receipt) -> float:
        if receipt is None:
            return 0.0
        return float(receipt.get("filled_qty", 0.0) or 0.0)

    def _receipt_remaining_qty(self, receipt) -> float:
        if receipt is None:
            return 0.0
        return float(receipt.get("remaining_volume", 0.0) or 0.0)

    def _load_entry_summary(self, yes_receipt: dict, no_receipt: dict) -> None:
        self.entry_yes_receipt = yes_receipt or {}
        self.entry_no_receipt = no_receipt or {}

        self.entry_yes_px = self.entry_yes_receipt.get("fill_price") or self.entry_yes_receipt.get("target_price")
        self.entry_no_px = self.entry_no_receipt.get("fill_price") or self.entry_no_receipt.get("target_price")

        self.yes_filled_qty = self._receipt_filled_qty(self.entry_yes_receipt)
        self.no_filled_qty = self._receipt_filled_qty(self.entry_no_receipt)

        self.matched_qty = min(self.yes_filled_qty, self.no_filled_qty)
        self.net_yes_qty = self.yes_filled_qty - self.no_filled_qty

        if self.entry_yes_px is not None and self.entry_no_px is not None:
            self.clean_pnl = round(self.matched_qty * (self.entry_no_px - self.entry_yes_px), 4)
        else:
            self.clean_pnl = 0.0

    # ============================================================
    # Rescue / pegging helpers
    # ============================================================
    def _remaining_rescue_qty(self) -> float:
        return max(0.0, self.rescue_qty_target - self.rescue_qty_filled)

    def _sync_rescue_progress(self, engine: KalshiMatchingEngine) -> None:
        """
        Count newly filled quantity on the current rescue order without double counting.
        This handles partial rescue fills before a cancel/reprice.
        """
        if self.rescue_side not in ["yes", "no"]:
            return

        order = engine.active_orders.get(self.rescue_side)
        if order is None:
            return

        order_id = order.get("order_id")
        if order_id is None:
            return

        total_filled_on_order = float(order.get("filled_qty", 0.0) or 0.0)
        already_counted = float(self.rescue_order_counted_qty.get(order_id, 0.0))
        delta_qty = total_filled_on_order - already_counted

        if delta_qty <= EPS:
            return

        delta_qty = min(delta_qty, self._remaining_rescue_qty())
        exit_px = float(order["order_price"])

        if self.net_yes_qty > 0:
            # Long YES from excess YES bid fills; sell YES to exit.
            pnl_delta = delta_qty * (exit_px - self.entry_yes_px)
        else:
            # Short YES from excess NO/ask fills; buy YES to exit.
            pnl_delta = delta_qty * (self.entry_no_px - exit_px)

        self.rescue_qty_filled += delta_qty
        self.rescue_pnl += pnl_delta
        self.rescue_exit_notional += delta_qty * exit_px
        self.rescue_order_counted_qty[order_id] = already_counted + delta_qty

        receipt = order.get("receipt", {})
        self.rescue_last_fill_ts = receipt.get("last_fill_ts") or receipt.get("fill_ts") or self.rescue_last_fill_ts

    def _current_rescue_order_is_good_enough(self, engine: KalshiMatchingEngine) -> bool:
        order = engine.active_orders.get(self.rescue_side)
        if not is_live_order(order):
            return False

        best_bid = engine.get_best_bid()
        best_ask = engine.get_best_ask()

        if best_bid is None or best_ask is None:
            return True

        current_px = float(order["order_price"])

        if self.rescue_side == "no":
            # Selling YES. If we are at or better than current best ask, keep.
            return current_px <= best_ask

        if self.rescue_side == "yes":
            # Buying YES. If we are at or better than current best bid, keep.
            return current_px >= best_bid

        return False

    def _compute_rescue_target_price(self, engine: KalshiMatchingEngine) -> Optional[float]:
        best_bid = engine.get_best_bid()
        best_ask = engine.get_best_ask()

        if best_bid is None or best_ask is None:
            return None

        best_bid = float(best_bid)
        best_ask = float(best_ask)

        if self.rescue_side == "no":
            # Need to sell YES. Join/improve ask side.
            target_px = round(best_ask - 0.01, 2)
            if target_px <= best_bid:
                target_px = round(best_bid + 0.01, 2)

        elif self.rescue_side == "yes":
            # Need to buy YES. Join/improve bid side.
            target_px = round(best_bid + 0.01, 2)
            if target_px >= best_ask:
                target_px = round(best_ask - 0.01, 2)

        else:
            return None

        return min(0.99, max(0.01, round(target_px, 2)))

    def _place_or_reprice_rescue_order(
        self,
        engine: KalshiMatchingEngine,
        current_ts: float,
        force: bool = False,
    ) -> bool:
        self._sync_rescue_progress(engine)

        remaining_qty = self._remaining_rescue_qty()
        if remaining_qty <= EPS:
            return True

        if not force and self._current_rescue_order_is_good_enough(engine):
            return True

        # Cancel existing live rescue order, but count any partial fills first.
        engine.cancel_active_order(self.rescue_side, current_ts)

        target_px = self._compute_rescue_target_price(engine)
        if target_px is None:
            return False

        engine.submit_limit_order(
            side=self.rescue_side,
            price=target_px,
            arrival_ts=current_ts,
            signal_ts=current_ts,
            volume=remaining_qty,
            tag="rescue_peg",
        )

        return True

    def _market_exit_remaining_inventory(
        self,
        engine: KalshiMatchingEngine,
        current_ts: float,
        reason: str,
    ) -> None:
        self._sync_rescue_progress(engine)

        if self.rescue_side in ["yes", "no"]:
            engine.cancel_active_order(self.rescue_side, current_ts)

        remaining_qty = self._remaining_rescue_qty()

        if remaining_qty > EPS:
            if self.net_yes_qty > 0:
                # Long YES. Sell at bid.
                exit_px = engine.get_best_bid()
                if exit_px is None:
                    exit_px = 0.01
                exit_px = round(float(exit_px), 2)
                pnl_delta = remaining_qty * (exit_px - self.entry_yes_px)

            else:
                # Short YES. Buy at ask.
                exit_px = engine.get_best_ask()
                if exit_px is None:
                    exit_px = 0.99
                exit_px = round(float(exit_px), 2)
                pnl_delta = remaining_qty * (self.entry_no_px - exit_px)

            self.rescue_qty_filled += remaining_qty
            self.rescue_pnl += pnl_delta
            self.rescue_exit_notional += remaining_qty * exit_px
            self.rescue_last_fill_ts = current_ts

        self._log_terminal_trade(
            outcome=self._classify_rescue_outcome(reason),
            exit_reason=reason,
            exit_ts=current_ts,
        )

    def _classify_rescue_outcome(self, label: str) -> str:
        total = self.clean_pnl + self.rescue_pnl

        if total > EPS:
            prefix = "Win"
        elif total < -EPS:
            prefix = "Loss"
        else:
            prefix = "Scratch"

        return f"{prefix} - {label}"

    # ============================================================
    # Logging
    # ============================================================
    def _log_scratch(self, reason: str):
        sig = self.active_signal or {}

        self.completed_trades.append({
            "ticker": sig.get("ticker"),
            "signal_side": sig.get("predicted_side", "maker"),
            "confidence": sig.get("confidence"),
            "signal_ts": sig.get("local_recv_ts"),
            "order_size": self.ORDER_SIZE,

            "outcome": reason,
            "exit_reason": reason,

            "entry_price": None,
            "exit_price": None,
            "gross_pnl": 0.0,

            "yes_filled_qty": 0.0,
            "no_filled_qty": 0.0,
            "matched_qty": 0.0,
            "net_yes_qty": 0.0,
            "clean_pnl": 0.0,
            "rescue_qty_target": 0.0,
            "rescue_qty_filled": 0.0,
            "rescue_pnl": 0.0,

            "yes_remaining_canceled": None,
            "no_remaining_canceled": None,

            "yes_fill_ts": None,
            "no_fill_ts": None,
            "first_fill_ts": None,
            "last_fill_ts": None,
            "time_between_first_last_fill": None,
            "time_from_signal_to_first_fill": None,
            "time_from_order_to_first_fill": None,
            "time_in_trade": None,
            "exit_ts": None,
        })

        self._reset_fsm()

    def _log_terminal_trade(self, outcome: str, exit_reason: str, exit_ts: float):
        sig = self.active_signal or {}

        yes_r = self.entry_yes_receipt or {}
        no_r = self.entry_no_receipt or {}

        gross_pnl = round(self.clean_pnl + self.rescue_pnl, 4)

        first_fill_candidates = [
            yes_r.get("first_fill_ts"),
            no_r.get("first_fill_ts"),
        ]
        first_fill_candidates = [x for x in first_fill_candidates if x is not None]

        last_fill_candidates = [
            yes_r.get("last_fill_ts") or yes_r.get("fill_ts"),
            no_r.get("last_fill_ts") or no_r.get("fill_ts"),
            self.rescue_last_fill_ts,
        ]
        last_fill_candidates = [x for x in last_fill_candidates if x is not None]

        first_fill_ts = min(first_fill_candidates) if first_fill_candidates else None
        last_fill_ts = max(last_fill_candidates) if last_fill_candidates else None

        signal_ts = sig.get("local_recv_ts")

        order_arrival_candidates = [
            yes_r.get("arrival_ts"),
            no_r.get("arrival_ts"),
        ]
        order_arrival_candidates = [x for x in order_arrival_candidates if x is not None]
        first_order_arrival_ts = min(order_arrival_candidates) if order_arrival_candidates else None

        time_between = (
            last_fill_ts - first_fill_ts
            if first_fill_ts is not None and last_fill_ts is not None
            else None
        )

        time_from_signal = (
            first_fill_ts - signal_ts
            if first_fill_ts is not None and signal_ts is not None
            else None
        )

        time_from_order = (
            first_fill_ts - first_order_arrival_ts
            if first_fill_ts is not None and first_order_arrival_ts is not None
            else None
        )

        time_in_trade = (
            exit_ts - first_fill_ts
            if first_fill_ts is not None and exit_ts is not None
            else None
        )

        rescue_exit_vwap = (
            self.rescue_exit_notional / self.rescue_qty_filled
            if self.rescue_qty_filled > EPS
            else None
        )

        # For old compatibility, entry/exit price are the initial YES bid and NO/YES-ask.
        entry_price = self.entry_yes_px
        exit_price = self.entry_no_px if self.rescue_qty_target <= EPS else rescue_exit_vwap

        self.completed_trades.append({
            "ticker": sig.get("ticker"),
            "signal_side": sig.get("predicted_side", "maker"),
            "confidence": sig.get("confidence"),
            "signal_ts": signal_ts,
            "order_size": self.ORDER_SIZE,

            "outcome": outcome,
            "exit_reason": exit_reason,

            "entry_price": entry_price,
            "exit_price": exit_price,
            "gross_pnl": gross_pnl,

            "yes_entry_price": self.entry_yes_px,
            "no_entry_price": self.entry_no_px,

            "yes_filled_qty": round(self.yes_filled_qty, 6),
            "no_filled_qty": round(self.no_filled_qty, 6),
            "matched_qty": round(self.matched_qty, 6),
            "net_yes_qty": round(self.net_yes_qty, 6),

            "clean_pnl": round(self.clean_pnl, 4),
            "rescue_side": self.rescue_side,
            "rescue_qty_target": round(self.rescue_qty_target, 6),
            "rescue_qty_filled": round(self.rescue_qty_filled, 6),
            "rescue_exit_vwap": rescue_exit_vwap,
            "rescue_pnl": round(self.rescue_pnl, 4),

            "yes_remaining_canceled": self._receipt_remaining_qty(yes_r),
            "no_remaining_canceled": self._receipt_remaining_qty(no_r),

            "yes_initial_v_ahead": yes_r.get("initial_v_ahead"),
            "no_initial_v_ahead": no_r.get("initial_v_ahead"),
            "yes_trades_ahead_of_us": yes_r.get("trades_ahead_of_us"),
            "no_trades_ahead_of_us": no_r.get("trades_ahead_of_us"),
            "yes_pulls_while_waiting": yes_r.get("pulls_while_waiting"),
            "no_pulls_while_waiting": no_r.get("pulls_while_waiting"),

            "yes_fill_ts": yes_r.get("first_fill_ts") or yes_r.get("fill_ts"),
            "no_fill_ts": no_r.get("first_fill_ts") or no_r.get("fill_ts"),
            "first_fill_ts": first_fill_ts,
            "last_fill_ts": last_fill_ts,
            "time_between_first_last_fill": time_between,
            "time_from_signal_to_first_fill": time_from_signal,
            "time_from_order_to_first_fill": time_from_order,
            "time_in_trade": time_in_trade,
            "exit_ts": exit_ts,
        })

        self._reset_fsm()

    def _reset_fsm(self):
        self.state = "WAITING_FOR_SIGNAL"
        self.active_signal = None

        self.whipsaw_start_ts = None
        self.peg_start_ts = None
        self.last_peg_ts = None

        self.entry_yes_receipt = None
        self.entry_no_receipt = None
        self.entry_yes_px = None
        self.entry_no_px = None

        self.yes_filled_qty = 0.0
        self.no_filled_qty = 0.0
        self.matched_qty = 0.0
        self.net_yes_qty = 0.0

        self.clean_pnl = 0.0

        self.rescue_side = None
        self.rescue_qty_target = 0.0
        self.rescue_qty_filled = 0.0
        self.rescue_pnl = 0.0
        self.rescue_exit_notional = 0.0
        self.rescue_last_fill_ts = None
        self.rescue_order_counted_qty = {}

In [ ]:
import os
import pandas as pd
from tqdm import tqdm
import time

# --- CONFIGURATION ---
PREDICTIONS_PATH = '/content/hot_density_signal_files/predictions_hot_epoch1_top700.parquet'
RAW_LOCAL_DIR = "/content/kalshi_clean_raw_v2"

ORDER_SIZES = [1, 5, 10, 75]

INCLUDE_OWN_ORDERS_IN_BEST = False

print("=== INITIATING PARTIAL-FILL MAKER-MAKER SIMULATOR ===")

df_preds = pd.read_parquet(PREDICTIONS_PATH)

required_cols = {"ticker", "local_recv_ts"}
missing = required_cols - set(df_preds.columns)
if missing:
    raise ValueError(f"Predictions missing required columns: {missing}")

tickers = df_preds["ticker"].unique()
print(f"Loaded {len(df_preds)} signals across {len(tickers)} unique markets.")
print(f"Order sizes to simulate: {ORDER_SIZES}")

all_size_results = []
global_start_time = time.time()

for order_size in ORDER_SIZES:
    print("\n" + "=" * 80)
    print(f"SIMULATING ORDER_SIZE = {order_size}")
    print("=" * 80)

    all_receipts = []
    size_start_time = time.time()

    for ticker in tqdm(tickers, desc=f"Simulating Q={order_size}"):
        ticker_preds = df_preds[df_preds["ticker"] == ticker].copy()

        clean_ticker = ticker.replace("dense_", "").replace(".parquet", "")
        raw_file_path = os.path.join(
            RAW_LOCAL_DIR,
            f"clean_raw_{clean_ticker}.parquet"
        )

        if not os.path.exists(raw_file_path):
            print(f"\n[!] Raw data missing for {clean_ticker}: {raw_file_path}")
            continue

        df_raw = pd.read_parquet(raw_file_path)

        if df_raw.empty:
            print(f"\n[!] Empty raw data for {clean_ticker}. Skipping.")
            continue

        sort_cols = ["local_recv_ts"]
        if "seq" in df_raw.columns:
            sort_cols.append("seq")

        df_raw = df_raw.sort_values(
            by=sort_cols,
            kind="mergesort"
        ).reset_index(drop=True)

        engine = KalshiMatchingEngine(
            include_own_orders_in_best=INCLUDE_OWN_ORDERS_IN_BEST
        )

        executor = KalshiExecutor(
            predictions_df=ticker_preds,
            order_size=order_size,
        )

        for row in df_raw.to_dict("records"):
            engine.process_tick(row)
            executor.update(row, engine)

        all_receipts.extend(executor.completed_trades)

    size_end_time = time.time()
    df_audit = pd.DataFrame(all_receipts)

    output_path = f"trade_receipts_Q{order_size}.parquet"

    print("\n--- SIZE RUN COMPLETE ---")
    print(f"ORDER_SIZE: {order_size}")
    print(f"Execution Time: {round(size_end_time - size_start_time, 2)} seconds")

    if df_audit.empty:
        print("[!] No trades were logged. Check FSM logic or threshold constraints.")
        continue

    df_audit.to_parquet(output_path, index=False)
    print(f"Saved audit log to {output_path}")

    print(f"\nTotal Signals Processed / Logged: {len(df_audit)}")

    print("\n--- OUTCOME DISTRIBUTION ---")
    print(df_audit["outcome"].value_counts(dropna=False))

    print("\n--- GROSS PNL SUMMARY ---")
    gross_pnl = df_audit["gross_pnl"].sum()
    print(f"Total Gross PnL: ${gross_pnl:.4f}")

    quantity_cols = [
        "yes_filled_qty",
        "no_filled_qty",
        "matched_qty",
        "net_yes_qty",
        "rescue_qty_target",
        "rescue_qty_filled",
        "clean_pnl",
        "rescue_pnl",
        "gross_pnl",
    ]

    existing_quantity_cols = [c for c in quantity_cols if c in df_audit.columns]

    print("\n--- QUANTITY / PNL TOTALS ---")
    print(df_audit[existing_quantity_cols].sum(numeric_only=True))

    print("\n--- EXIT REASON DISTRIBUTION ---")
    if "exit_reason" in df_audit.columns:
        print(df_audit["exit_reason"].value_counts(dropna=False))

    print("\n--- SAMPLE NON-SCRATCH TRADES ---")
    non_scratch = df_audit[
        ~df_audit["outcome"].astype(str).str.contains(
            "Skipped|Dead Market|No Fills",
            na=False
        )
    ]

    if not non_scratch.empty:
        display_cols = [
            "ticker",
            "outcome",
            "order_size",
            "gross_pnl",
            "yes_filled_qty",
            "no_filled_qty",
            "matched_qty",
            "net_yes_qty",
            "clean_pnl",
            "rescue_pnl",
            "rescue_qty_target",
            "rescue_qty_filled",
            "time_in_trade",
        ]
        display_cols = [c for c in display_cols if c in non_scratch.columns]
        print(non_scratch[display_cols].head(20))
    else:
        print("No non-scratch trades.")

    all_size_results.append({
        "order_size": order_size,
        "num_logged": len(df_audit),
        "gross_pnl": gross_pnl,
        "matched_qty": df_audit["matched_qty"].sum() if "matched_qty" in df_audit.columns else None,
        "net_abs_qty": df_audit["net_yes_qty"].abs().sum() if "net_yes_qty" in df_audit.columns else None,
        "rescue_qty_target": df_audit["rescue_qty_target"].sum() if "rescue_qty_target" in df_audit.columns else None,
        "rescue_qty_filled": df_audit["rescue_qty_filled"].sum() if "rescue_qty_filled" in df_audit.columns else None,
    })

global_end_time = time.time()

print("\n" + "=" * 80)
print("ALL SIZE RUNS COMPLETE")
print(f"Total Execution Time: {round(global_end_time - global_start_time, 2)} seconds")
print("=" * 80)

if all_size_results:
    df_size_summary = pd.DataFrame(all_size_results)
    df_size_summary.to_parquet("order_size_summary.parquet", index=False)

    print("\n--- ORDER SIZE SUMMARY ---")
    print(df_size_summary)
else:
    print("[!] No size results generated.")

=== INITIATING PARTIAL-FILL MAKER-MAKER SIMULATOR ===
Loaded 700 signals across 18 unique markets.
Order sizes to simulate: [1, 5, 10, 75]

SIMULATING ORDER_SIZE = 1


Simulating Q=1: 100%|██████████| 18/18 [00:54<00:00,  3.01s/it]



--- SIZE RUN COMPLETE ---
ORDER_SIZE: 1
Execution Time: 54.27 seconds
Saved audit log to trade_receipts_Q1.parquet

Total Signals Processed / Logged: 700

--- OUTCOME DISTRIBUTION ---
outcome
Skipped - Stale Signal                        297
Skipped - Spread < 2c                         277
Win - Full Clean Whipsaw                       63
Loss - Loss/Scratch - 1c Spread Bailout        18
Scratch - Loss/Scratch - 1c Spread Bailout     13
Loss - Closed - Rescue Peg Filled               9
Win - Loss/Scratch - 1c Spread Bailout          8
Win - Closed - Rescue Peg Filled                7
Scratch - Dead Market, No Fills                 4
Scratch - Closed - Rescue Peg Filled            3
Skipped - Vacuum                                1
Name: count, dtype: int64

--- GROSS PNL SUMMARY ---
Total Gross PnL: $1.7500

--- QUANTITY / PNL TOTALS ---
yes_filled_qty       90.00
no_filled_qty        94.00
matched_qty          63.00
net_yes_qty          -4.00
rescue_qty_target    58.00
rescue_qty_fi

Simulating Q=5: 100%|██████████| 18/18 [00:54<00:00,  3.01s/it]



--- SIZE RUN COMPLETE ---
ORDER_SIZE: 5
Execution Time: 54.11 seconds
Saved audit log to trade_receipts_Q5.parquet

Total Signals Processed / Logged: 700

--- OUTCOME DISTRIBUTION ---
outcome
Skipped - Spread < 2c                         306
Skipped - Stale Signal                        267
Win - Full Clean Whipsaw                       62
Loss - Loss/Scratch - 1c Spread Bailout        19
Scratch - Loss/Scratch - 1c Spread Bailout     11
Win - Loss/Scratch - 1c Spread Bailout         10
Loss - Closed - Rescue Peg Filled               9
Scratch - Dead Market, No Fills                 7
Win - Closed - Rescue Peg Filled                6
Scratch - Closed - Rescue Peg Filled            2
Skipped - Vacuum                                1
Name: count, dtype: int64

--- GROSS PNL SUMMARY ---
Total Gross PnL: $7.7329

--- QUANTITY / PNL TOTALS ---
yes_filled_qty       441.0100
no_filled_qty        465.5600
matched_qty          315.5700
net_yes_qty          -24.5500
rescue_qty_target    275.430

Simulating Q=10: 100%|██████████| 18/18 [00:54<00:00,  3.04s/it]



--- SIZE RUN COMPLETE ---
ORDER_SIZE: 10
Execution Time: 54.68 seconds
Saved audit log to trade_receipts_Q10.parquet

Total Signals Processed / Logged: 700

--- OUTCOME DISTRIBUTION ---
outcome
Skipped - Spread < 2c                         310
Skipped - Stale Signal                        267
Win - Full Clean Whipsaw                       56
Loss - Loss/Scratch - 1c Spread Bailout        19
Win - Loss/Scratch - 1c Spread Bailout         12
Scratch - Loss/Scratch - 1c Spread Bailout     11
Loss - Closed - Rescue Peg Filled               9
Win - Closed - Rescue Peg Filled                7
Scratch - Dead Market, No Fills                 6
Scratch - Closed - Rescue Peg Filled            2
Skipped - Vacuum                                1
Name: count, dtype: int64

--- GROSS PNL SUMMARY ---
Total Gross PnL: $15.5286

--- QUANTITY / PNL TOTALS ---
yes_filled_qty       846.6400
no_filled_qty        904.1500
matched_qty          595.7900
net_yes_qty          -57.5100
rescue_qty_target    559.

Simulating Q=75: 100%|██████████| 18/18 [00:54<00:00,  3.03s/it]


--- SIZE RUN COMPLETE ---
ORDER_SIZE: 75
Execution Time: 54.51 seconds
Saved audit log to trade_receipts_Q75.parquet

Total Signals Processed / Logged: 700

--- OUTCOME DISTRIBUTION ---
outcome
Skipped - Spread < 2c                         296
Skipped - Stale Signal                        292
Win - Full Clean Whipsaw                       45
Loss - Loss/Scratch - 1c Spread Bailout        22
Scratch - Loss/Scratch - 1c Spread Bailout     11
Win - Loss/Scratch - 1c Spread Bailout         11
Win - Closed - Rescue Peg Filled                8
Loss - Closed - Rescue Peg Filled               7
Scratch - Dead Market, No Fills                 4
Scratch - Closed - Rescue Peg Filled            3
Skipped - Vacuum                                1
Name: count, dtype: int64

--- GROSS PNL SUMMARY ---
Total Gross PnL: $69.9969

--- QUANTITY / PNL TOTALS ---
yes_filled_qty       5472.3950
no_filled_qty        5951.0100
matched_qty          3604.4050
net_yes_qty          -478.6150
rescue_qty_target    